# GoldenCheetah 데이터 탐색

## 분석 목적

GoldenCheetah 공개 데이터가 훈련 분석과 오늘의 훈련 추천 프로젝트에
사용할 수 있는지 확인한다.

## 현재까지 확인한 내용

- 전체 운동 기록: 731개
- 자전거 운동: 592개
- 파워 센서 포함: 469개
- 심박 센서 포함: 465개
- 파워와 심박 모두 포함: 373개
- 라이딩마다 센서와 요약 지표 구성이 다름

## 실행 환경과 파일 경로 설정

분석에 필요한 라이브러리를 불러오고 현재 Python 환경과 프로젝트
경로를 확인한다. 원본 GoldenCheetah JSON 파일의 경로도 이곳에서
설정한다.


In [2]:
import sys
from pathlib import Path

import json

import pandas as pd

project_root = Path("..").resolve()

print("Python 경로:", sys.executable)
print("프로젝트 경로:", project_root)
print("pandas 버전:", pd.__version__)

json_path = (
    project_root
    / "data"
    / "raw"
    / "033874ce-e20d-44ba-9cc9-125030b6662f"
    / "{033874ce-e20d-44ba-9cc9-125030b6662f}.json"
)

Python 경로: /Users/hooni/Documents/ChatGPT/Cycling App/.venv/bin/python
프로젝트 경로: /Users/hooni/Documents/ChatGPT/Cycling App
pandas 버전: 3.0.5


## 1. JSON 데이터 불러오기

JSON 파일을 Python 자료형으로 불러온다.
전체 운동 중 `Bike` 기록만 선택하여 pandas 분석에 사용한다.

In [3]:
with json_path.open("r", encoding="utf-8") as file:
    cycling_data = json.load(file)

rides = cycling_data["RIDES"]

bike_rides = [ride for ride in rides if ride["sport"] == "Bike"]

print("전체 운동 수:", len(rides))
print("자전거 운동 수:", len(bike_rides))

전체 운동 수: 731
자전거 운동 수: 592


라이드 기록의 `data`는 15자리의 대문자 알파벳 문자열을 값으로 가지는데, 해당 운동에 어떤 센서 데이터가 포함되어 있는지 나타낸다.

| 문자 | 데이터 |
|---|---|
| `T` | 시간 |
| `D` | 거리 |
| `S` | 속도 |
| `P` | 파워 |
| `H` | 심박수 |
| `C` | 케이던스 |
| `N` | 토크 |
| `A` | 고도 |
| `G` | GPS |
| `L` | 경사도 |
| `W` | 풍속 |
| `E` | 온도 |
| `V` | 좌우 페달 데이터 |
| `O` | 근육 산소 관련 데이터 |
| `R` | Garmin 러닝 다이내믹스 |

In [4]:
first_ride = bike_rides[0]

print(first_ride.keys())
print("첫 라이드의 날짜", first_ride["date"])
print("첫 라이드 기록의 센서 목록:", first_ride["data"])

dict_keys(['date', 'data', 'sport', 'METRICS'])
첫 라이드의 날짜 2005/06/25 14:26:00 UTC
첫 라이드 기록의 센서 목록: TDS-H--A-L-----


첫 라이드의 경우 `TDS-H--A-L-----`로, 시간, 거리, 속도, 심박수, 고도, 경사도 데이터를 포함하고 있다.

## 2. 자전거 기록을 표로 변환

592개의 자전거 기록은 딕셔너리로 구성되어 있다.
pandas의 `DataFrame`을 이용하여 행과 열로 구성된 표로 변환한다.

In [5]:
bike_raw_df = pd.DataFrame(bike_rides)

print("자료형", type(bike_raw_df))
print("표 크기:", bike_raw_df.shape)
print("열 이름", bike_raw_df.columns)

bike_raw_df.head()

자료형 <class 'pandas.DataFrame'>
표 크기: (592, 5)
열 이름 Index(['date', 'data', 'sport', 'METRICS', 'XDATA'], dtype='str')


,date,data,sport,METRICS,XDATA
0,2005/06/25 14:26:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
1,2005/06/27 08:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
2,2005/06/28 17:44:00 UTC,TDS-HC-A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
3,2005/07/06 18:02:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
4,2005/07/10 09:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN


- 표는 592개의 행과 5개의 열로 구성되어 있다.
- 기본 열은 `date`, `data`, `sport`, `METRICS`, `XDATA`이다.
- `METRICS`는 아직 하나의 딕셔너리로 저장되어 있다.
- `XDATA`는 일부 라이딩에만 존재하므로 대부분 결측값으로 표시된다.

## 3. METRICS 펼치기

각 라이딩의 `METRICS`에는 운동 시간, 거리, 파워, 심박수 등
여러 요약 지표가 딕셔너리 형태로 저장되어 있다.

`METRICS`의 각 키를 DataFrame의 개별 열로 변환한다.

In [6]:
metrics_df = pd.json_normalize(bike_raw_df["METRICS"])

print("METRICS 표 크기", metrics_df.shape)
print("앞쪽 열 10개:")
print(metrics_df.columns[:10])

METRICS 표 크기 (592, 228)
앞쪽 열 10개:
Index(['a_skiba_variability_index', 'a_coggam_variability_index', 'ride_count',
       'workout_time', 'time_riding', 'total_distance', 'climb_rating',
       'athlete_weight', 'elevation_gain', 'elevation_loss'],
      dtype='str')


In [7]:
sample_columns = [
    "workout_time",
    'time_riding',
    "total_distance",
    'elevation_gain',
    "average_power",
    "average_hr",
    "coggan_tss",
    "coggan_if",
]

metrics_df[sample_columns].head()

,workout_time,time_riding,total_distance,elevation_gain,average_power,average_hr,coggan_tss,coggan_if
0,4800.00000,4780.00000,35.32750,367.00000,NaN,"[146.91667, 960.00000]",NaN,NaN
1,6476.00000,6325.00000,35.01400,467.50000,NaN,"[124.97267, 6476.00000]",NaN,NaN
2,2156.00000,2156.00000,17.65300,92.00000,NaN,"[157.04592, 2156.00000]",NaN,NaN
3,6360.00000,6320.00000,31.04950,512.00000,NaN,"[120.05660, 1272.00000]",NaN,NaN
4,4680.00000,4660.00000,32.77000,471.00000,NaN,"[146.42735, 936.00000]",NaN,NaN


- 592개 자전거 기록의 `METRICS`를 펼치자 228개의 지표가 나타났다.
- 라이딩마다 포함된 지표가 달라 전체 지표 수가 많아졌다.
- 이전 .py 파일에서 보았듯이, 평균 파워와 평균 심박은 일부 행에서 리스트 형태로 저장되어 있다.
- 파워 센서가 없는 초기 라이딩에서는 파워, TSS, IF가 결측값으로 표시된다.
- 228개 지표를 모두 사용하지 않고 분석 목적에 필요한 지표만 선택해야 한다.

## 4. 분석에 사용할 지표 선택

228개의 `METRICS` 지표들은 모두 서로 다른 센서 데이터가 아니라, 기본 센서값을
여러 계산 방식으로 요약한 지표도 포함하고 있다.

예를 들어 파워 관련 지표에는 다음과 같은 계산 계열이 존재한다.

- `coggan_`: Coggan 방식의 파워 강도와 훈련 부하 지표
- `skiba_`: Skiba 방식의 파워 강도와 훈련 부하 지표
- `a_`: 고도의 영향을 반영한 보정 지표

비슷한 의미의 지표를 모두 사용하면 정보가 중복될 수 있으므로,
첫 번째 분석에서는 해석하기 쉬운 기본 지표와 Coggan 계열을 우선 사용한다.

### 기본 정보

- `date`: 운동 날짜
- `data`: 기록된 센서 종류
- `sport`: 운동 종류

### 운동량

- `workout_time`: 전체 운동 시간
- `time_riding`: 실제 이동 시간
- `total_distance`: 총거리
- `elevation_gain`: 누적 상승고도
- `average_speed`: 평균 속도

### 파워

- `average_power`: 평균 파워
- `coggan_np`: 변동성을 고려한 대표 파워
- `max_power`: 최대 파워
- `cp_setting`: 해당 시점의 기준 파워

### 심박과 케이던스

- `average_hr`: 평균 심박수
- `max_heartrate`: 최대 심박수
- `average_cad`: 평균 케이던스
- `max_cadence`: 최대 케이던스

### 훈련 강도와 부하

- `coggan_if`: 기준 파워 대비 운동 강도
- `coggan_tss`: 운동 시간과 강도를 반영한 훈련 부하

`skiba_` 계열과 `a_` 계열은 원본에 보존하고, 이후 계산 방식과
고도 영향을 비교할 필요가 생기면 별도로 분석한다.

In [8]:
selected_metric_columns = [
    "workout_time",     # 전체 운동 시간
    "time_riding",      # 실제 이동 시간
    "total_distance",   # 총거리
    "elevation_gain",   # 획득 고도
    "average_speed",    # 평균 속도
    "average_power",    # 평균 파워
    "max_power",        # 최대 파워
    "coggan_np",        # NP
    "cp_setting",       # 해당 시점의 기준 파워
    "average_hr",       # 평균 심박수
    "max_heartrate",    # 최대 심박수
    "average_cad",      # 평균 케이던스
    "max_cadence",      # 최대 케이던스
    "coggan_if",        # IF
    "coggan_tss",       # TSS
]

selected_metrics_df = metrics_df[selected_metric_columns].copy()

print(selected_metrics_df.shape)
selected_metrics_df.head()

(592, 15)


,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,4800.00000,4780.00000,35.32750,367.00000,26.60649,NaN,NaN,NaN,200.00000,"[146.91667, 960.00000]",184.00000,NaN,NaN,NaN,NaN
1,6476.00000,6325.00000,35.01400,467.50000,19.92892,NaN,NaN,NaN,200.00000,"[124.97267, 6476.00000]",182.00000,NaN,NaN,NaN,NaN
2,2156.00000,2156.00000,17.65300,92.00000,29.71052,NaN,NaN,NaN,200.00000,"[157.04592, 2156.00000]",168.00000,"[88.72929, 1919.00000]",98.00000,NaN,NaN
3,6360.00000,6320.00000,31.04950,512.00000,17.68642,NaN,NaN,NaN,200.00000,"[120.05660, 1272.00000]",182.00000,NaN,NaN,NaN,NaN
4,4680.00000,4660.00000,32.77000,471.00000,25.31588,NaN,NaN,NaN,200.00000,"[146.42735, 936.00000]",177.00000,NaN,NaN,NaN,NaN


In [9]:
metadata_df = bike_raw_df[
    ["data", "date", "sport"]
].copy()

analysis_df = pd.concat(
    [metadata_df, selected_metrics_df],
    axis=1
)

print(analysis_df.shape)
analysis_df.head()

(592, 18)


,data,date,sport,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,TDS-H--A-L-----,2005/06/25 14:26:00 UTC,Bike,4800.00000,4780.00000,35.32750,367.00000,26.60649,NaN,NaN,NaN,200.00000,"[146.91667, 960.00000]",184.00000,NaN,NaN,NaN,NaN
1,TDS-H--A-L-----,2005/06/27 08:39:00 UTC,Bike,6476.00000,6325.00000,35.01400,467.50000,19.92892,NaN,NaN,NaN,200.00000,"[124.97267, 6476.00000]",182.00000,NaN,NaN,NaN,NaN
2,TDS-HC-A-L-----,2005/06/28 17:44:00 UTC,Bike,2156.00000,2156.00000,17.65300,92.00000,29.71052,NaN,NaN,NaN,200.00000,"[157.04592, 2156.00000]",168.00000,"[88.72929, 1919.00000]",98.00000,NaN,NaN
3,TDS-H--A-L-----,2005/07/06 18:02:00 UTC,Bike,6360.00000,6320.00000,31.04950,512.00000,17.68642,NaN,NaN,NaN,200.00000,"[120.05660, 1272.00000]",182.00000,NaN,NaN,NaN,NaN
4,TDS-H--A-L-----,2005/07/10 09:39:00 UTC,Bike,4680.00000,4660.00000,32.77000,471.00000,25.31588,NaN,NaN,NaN,200.00000,"[146.42735, 936.00000]",177.00000,NaN,NaN,NaN,NaN


## 5. 지표의 자료형 확인과 변환

선택한 지표의 실제 값을 확인한 결과, 숫자가 다음 두 가지 형태로
저장되어 있었다.

- 숫자를 나타내는 문자열: `"4316.00000"`
- 지표값과 관측 정보를 담은 리스트: `["145.75371", "4316.00000"]`

리스트의 첫 번째 요소는 분석에 사용할 지표값이고, 두 번째 요소는
평균 계산에 사용된 관측값 수 또는 가중치 정보로 보인다.

`average_hr`에서는 대부분의 값이 리스트였지만 3개는 문자열이었다.
이 3개 라이드는 센서 정보를 나타내는 `data`가 비어 있고 최대 심박수도
기록되어 있지 않았다. 따라서 센서 시계열에서 계산된 값이 아니라
수동으로 입력했거나 외부에서 가져온 요약값일 가능성이 높다.

원본 구조를 보존하기 위해 `analysis_df`는 변경하지 않는다.
대신 `cleaned_df`를 복사하여 다음과 같이 변환한다.

- 리스트는 첫 번째 요소를 사용한다.
- 숫자 형태의 문자열은 실제 숫자로 변환한다.
- 변환할 수 없는 값은 결측값으로 처리한다.
- 리스트의 두 번째 요소는 필요할 경우 원본 데이터에서 다시 확인한다.

In [10]:
print(analysis_df.dtypes)

data                 str
date                 str
sport                str
workout_time         str
time_riding          str
total_distance       str
elevation_gain       str
average_speed        str
average_power     object
max_power            str
coggan_np         object
cp_setting           str
average_hr        object
max_heartrate        str
average_cad       object
max_cadence          str
coggan_if         object
coggan_tss           str
dtype: object


In [11]:
for column in selected_metric_columns:
    values = analysis_df[column].dropna()
    first_value = values.iloc[0]

    print(
        column,
        "| 값:", first_value,
        "| 자료형:", type(first_value).__name__
    )

workout_time | 값: 4800.00000 | 자료형: str
time_riding | 값: 4780.00000 | 자료형: str
total_distance | 값: 35.32750 | 자료형: str
elevation_gain | 값: 367.00000 | 자료형: str
average_speed | 값: 26.60649 | 자료형: str
average_power | 값: ['188.85205', '12808.00000'] | 자료형: list
max_power | 값: 611.00000 | 자료형: str
coggan_np | 값: ['237.99715', '16138.08000'] | 자료형: list
cp_setting | 값: 200.00000 | 자료형: str
average_hr | 값: ['146.91667', '960.00000'] | 자료형: list
max_heartrate | 값: 184.00000 | 자료형: str
average_cad | 값: ['88.72929', '1919.00000'] | 자료형: list
max_cadence | 값: 98.00000 | 자료형: str
coggan_if | 값: ['0.86544', '16138.08000'] | 자료형: list
coggan_tss | 값: 335.75887 | 자료형: str


In [12]:
for column in selected_metric_columns:
    type_counts = (
        analysis_df[column]
        .dropna()
        .map(type)
        .value_counts()
    )

    print(f"{type_counts}\n")

workout_time
<class 'str'>    591
Name: count, dtype: int64

time_riding
<class 'str'>    574
Name: count, dtype: int64

total_distance
<class 'str'>    563
Name: count, dtype: int64

elevation_gain
<class 'str'>    496
Name: count, dtype: int64

average_speed
<class 'str'>    563
Name: count, dtype: int64

average_power
<class 'list'>    469
Name: count, dtype: int64

max_power
<class 'str'>    469
Name: count, dtype: int64

coggan_np
<class 'list'>    469
Name: count, dtype: int64

cp_setting
<class 'str'>    592
Name: count, dtype: int64

average_hr
<class 'list'>    465
<class 'str'>       3
Name: count, dtype: int64

max_heartrate
<class 'str'>    465
Name: count, dtype: int64

average_cad
<class 'list'>    532
Name: count, dtype: int64

max_cadence
<class 'str'>    532
Name: count, dtype: int64

coggan_if
<class 'list'>    470
Name: count, dtype: int64

coggan_tss
<class 'str'>    490
Name: count, dtype: int64



In [13]:
average_hr_is_string = analysis_df["average_hr"].map(type) == str

analysis_df.loc[
    average_hr_is_string,
    [
        "date",
        "data",
        "average_hr",
        "max_heartrate",
        "workout_time",
    ]
]

,date,data,average_hr,max_heartrate,workout_time
392,2012/01/04 15:35:51 UTC,---------------,148.00000,NaN,4200.00000
393,2012/01/05 12:32:51 UTC,---------------,145.00000,NaN,5820.00000
394,2012/01/06 13:22:53 UTC,---------------,145.00000,NaN,2700.00000


### 자료형 확인 결과

`average_hr`의 유효한 값 대부분은 리스트이지만, 3개는 문자열로
저장되어 있다.

해당 3개 라이드는 `data`에 센서 기록이 표시되어 있지 않고
최대 심박수도 비어 있다. 따라서 시계열 심박 데이터에서 계산된 값이
아니라, 수동으로 입력했거나 외부에서 가져온 요약값일 가능성이 높다.

지금은 삭제하지 않고 이후 데이터 품질을 분류할 때 별도로 구분한다.

In [14]:
def extract_metric_value(value):
    if isinstance(value, list):
        value = value[0]

    return pd.to_numeric(value, errors="coerce")

In [15]:
cleaned_df = analysis_df.copy()

for column in selected_metric_columns:
    cleaned_df[column] = cleaned_df[column].map(
        extract_metric_value
    )

print(cleaned_df.dtypes)
display(cleaned_df.head())

data                  str
date                  str
sport                 str
workout_time      float64
time_riding       float64
total_distance    float64
elevation_gain    float64
average_speed     float64
average_power     float64
max_power         float64
coggan_np         float64
cp_setting        float64
average_hr        float64
max_heartrate     float64
average_cad       float64
max_cadence       float64
coggan_if         float64
coggan_tss        float64
dtype: object


,data,date,sport,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,TDS-H--A-L-----,2005/06/25 14:26:00 UTC,Bike,4800.0,4780.0,35.3275,367.0,26.60649,NaN,NaN,NaN,200.0,146.91667,184.0,NaN,NaN,NaN,NaN
1,TDS-H--A-L-----,2005/06/27 08:39:00 UTC,Bike,6476.0,6325.0,35.0140,467.5,19.92892,NaN,NaN,NaN,200.0,124.97267,182.0,NaN,NaN,NaN,NaN
2,TDS-HC-A-L-----,2005/06/28 17:44:00 UTC,Bike,2156.0,2156.0,17.6530,92.0,29.71052,NaN,NaN,NaN,200.0,157.04592,168.0,88.72929,98.0,NaN,NaN
3,TDS-H--A-L-----,2005/07/06 18:02:00 UTC,Bike,6360.0,6320.0,31.0495,512.0,17.68642,NaN,NaN,NaN,200.0,120.05660,182.0,NaN,NaN,NaN,NaN
4,TDS-H--A-L-----,2005/07/10 09:39:00 UTC,Bike,4680.0,4660.0,32.7700,471.0,25.31588,NaN,NaN,NaN,200.0,146.42735,177.0,NaN,NaN,NaN,NaN


## 6. 선택한 지표의 결측값 확인

숫자로 변환한 `cleaned_df`를 이용하여 각 후보 지표의 결측 개수와
결측 비율을 확인한다.

결측값의 분포를 통해 실제 분석에 사용할 수 있는 지표와
별도의 품질 기준이 필요한 지표를 판단한다.

In [16]:
missing_count = cleaned_df[selected_metric_columns].isna().sum()
missing_ratio = cleaned_df[selected_metric_columns].isna().mean() * 100

missing_summary_df = pd.DataFrame({
    "missing_count": missing_count,
    "missing_ratio": missing_ratio,
})

missing_summary_df.round(2)

,missing_count,missing_ratio
workout_time,1,0.17
time_riding,18,3.04
total_distance,29,4.90
elevation_gain,96,16.22
average_speed,29,4.90
average_power,123,20.78
max_power,123,20.78
coggan_np,123,20.78
cp_setting,0,0.00
average_hr,124,20.95


### 결측값 확인 결과

파워와 심박 관련 지표는 전체 자전거 기록의 약 20%에서 결측값으로
나타났다. 모든 라이딩에 파워 미터와 심박 센서가 사용된 것은 아니기
때문으로 보인다.

IF도 파워 지표와 비슷한 결측 비율을 보였다. 반면 TSS는 파워보다
결측 비율이 낮았다. 일부 TSS는 파워 센서 데이터로부터 자동 계산된
값이 아니라 수동으로 입력했거나 별도의 방식으로 추정한 값일 가능성이
있다.

따라서 결측값이 있는 라이드를 모두 삭제하지 않고, 분석 목적과
센서 유무에 따라 사용할 라이드를 구분할 필요가 있다.

## 7. 센서 표시와 실제 지표 비교

`data`에 표시된 파워 및 심박 센서의 유무와 실제 요약 지표의
존재 여부가 일치하는지 확인한다.

이를 통해 센서 데이터에서 계산된 값과 수동으로 입력된 것으로 보이는
요약값을 구분할 기준을 마련한다.

In [17]:
cleaned_df["has_power_sensor"] = (
    cleaned_df["data"].str.contains("P", na=False)
)

cleaned_df["has_hr_sensor"] = (
    cleaned_df["data"].str.contains("H", na=False)
)

sensor_metric_summary_df = pd.DataFrame(
    {
        "sensor_count": [
            cleaned_df["has_power_sensor"].sum(),
            cleaned_df["has_hr_sensor"].sum(),
        ],
        "metric_count": [
            cleaned_df["average_power"].notna().sum(),
            cleaned_df["average_hr"].notna().sum(),
        ],
    },
    index=["power", "heart_rate"],
)

sensor_metric_summary_df

,sensor_count,metric_count
power,469,469
heart_rate,465,468


In [18]:
cleaned_df["has_power_metric"] = (
    cleaned_df["average_power"].notna()
)

cleaned_df["has_hr_metric"] = (
    cleaned_df["average_hr"].notna()
)

power_mismatch = (
    cleaned_df["has_power_sensor"]
    != cleaned_df["has_power_metric"]
)

hr_mismatch = (
    cleaned_df["has_hr_sensor"]
    != cleaned_df["has_hr_metric"]
)

print("파워 불일치:", power_mismatch.sum())
print("심박 불일치:", hr_mismatch.sum())

파워 불일치: 0
심박 불일치: 3


In [19]:
cleaned_df.loc[
    hr_mismatch,
    [
        "date",
        "data",
        "average_hr",
        "max_heartrate",
        "has_hr_sensor",
        "has_hr_metric",
    ]
]

,date,data,average_hr,max_heartrate,has_hr_sensor,has_hr_metric
392,2012/01/04 15:35:51 UTC,---------------,148.0,NaN,False,True
393,2012/01/05 12:32:51 UTC,---------------,145.0,NaN,False,True
394,2012/01/06 13:22:53 UTC,---------------,145.0,NaN,False,True


### 센서와 지표 비교 결과

파워 센서가 표시된 라이드는 469개이며, 평균 파워가 존재하는
라이드도 469개였다. 행 단위로 비교한 결과 불일치는 없었다.

심박 센서가 표시된 라이드는 465개이지만, 평균 심박이 존재하는
라이드는 468개였다. 행 단위 비교 결과 3개의 불일치가 확인되었다.

이 3개 라이드는 심박 센서 표시 없이 평균 심박만 존재하며, 앞서 확인한
문자열 형태의 평균 심박 기록과 일치한다. 따라서 센서 시계열에서 계산된
값이 아니라 수동 입력 또는 외부에서 가져온 요약값으로 판단한다.

## 8. 날짜 자료형 변환

현재 `date`는 문자열로 저장되어 있다. 시간순 정렬과 기간별 훈련
흐름 분석을 위해 pandas의 날짜 자료형으로 변환한다.

원본 날짜는 UTC 기준이므로 변환 후에도 UTC 시간대를 유지한다.

In [20]:
cleaned_df["date"] = pd.to_datetime(
    cleaned_df["date"],
    format="%Y/%m/%d %H:%M:%S UTC",
    utc=True,
    errors="coerce",
)

print("자료형:", cleaned_df["date"].dtype)
print("변환 실패:", cleaned_df["date"].isna().sum())
print("첫 운동:", cleaned_df["date"].min())
print("마지막 운동:", cleaned_df["date"].max())

자료형: datetime64[us, UTC]
변환 실패: 0
첫 운동: 2005-06-25 14:26:00+00:00
마지막 운동: 2017-07-18 11:01:20+00:00


### 날짜 변환 결과

592개 자전거 기록의 날짜가 모두 정상적으로 변환되었으며,
변환에 실패한 기록은 없었다.

데이터는 2005년 6월 25일부터 2017년 7월 18일까지 약 12년의
운동 기록을 포함한다. 이후 날짜를 기준으로 정렬하고 주간·월간
훈련 흐름을 분석할 수 있다.

### 날짜순 정렬

날짜 변환 후 기록이 시간순으로 정렬되어 있는지 확인한다.
이후 기간별 훈련 흐름을 올바르게 계산할 수 있도록 날짜를 기준으로
오름차순 정렬하고 행 인덱스를 다시 설정한다.

In [21]:
print(
    "현재 날짜순 정렬:",
    cleaned_df["date"].is_monotonic_increasing
)

현재 날짜순 정렬: True


### 날짜순 정렬 확인

날짜가 오름차순으로 정렬되어 있는지 확인한 결과 `True`가
나타났다. 현재 데이터가 이미 과거부터 최근 순서로 정렬되어 있으므로
별도의 정렬은 수행하지 않는다.

## 9. 센서 없는 심박 요약 기록 표시

심박 센서 표시는 없지만 평균 심박 지표가 존재하는 라이드를
별도의 불리언 열로 표시한다.

이 기록은 수동 입력 또는 외부에서 가져온 요약값일 가능성이 있지만,
현재 데이터만으로 출처를 확정할 수 없으므로 관찰된 사실을 나타내는
이름을 사용한다.

In [22]:
cleaned_df["hr_metric_without_sensor"] = (
    ~cleaned_df["has_hr_sensor"]
    & cleaned_df["has_hr_metric"]
)

print(
    "센서 없는 심박 요약:",
    cleaned_df["hr_metric_without_sensor"].sum()
)

cleaned_df.loc[
    cleaned_df["hr_metric_without_sensor"],
    [
        "date",
        "data",
        "average_hr",
        "max_heartrate",
    ]
]

센서 없는 심박 요약: 3


,date,data,average_hr,max_heartrate
392,2012-01-04 15:35:51+00:00,---------------,148.0,NaN
393,2012-01-05 12:32:51+00:00,---------------,145.0,NaN
394,2012-01-06 13:22:53+00:00,---------------,145.0,NaN


### 품질 표시 결과

심박 센서 표시는 없지만 평균 심박이 존재하는 기록은 3개였다.
모두 기존에 확인한 문자열 형태의 평균 심박 기록과 일치하며,
최대 심박은 결측값으로 나타났다.

이 기록을 `hr_metric_without_sensor`로 표시하여 이후 센서 기반
심박 분석에서는 구분할 수 있도록 한다. 다만 수동 입력 여부는
확정하지 않는다.

## 10. 파워 센서 없는 TSS 기록 표시

파워 센서 표시는 없지만 TSS가 존재하는 라이드를 구분한다.
이 값은 파워 시계열에서 직접 계산된 값이 아닐 가능성이 있으므로
`tss_without_power_sensor` 품질 표시를 추가한다.

In [23]:
cleaned_df["has_tss_metric"] = (
    cleaned_df["coggan_tss"].notna()
)

cleaned_df["tss_without_power_sensor"] = (
    ~cleaned_df["has_power_sensor"]
    & cleaned_df["has_tss_metric"]
)

print(
    "파워 센서 없는 TSS:",
    cleaned_df["tss_without_power_sensor"].sum()
)

cleaned_df.loc[
    cleaned_df["tss_without_power_sensor"],
    [
        "date",
        "data",
        "average_power",
        "coggan_np",
        "coggan_if",
        "coggan_tss",
    ],
]

파워 센서 없는 TSS: 21


,date,data,average_power,coggan_np,coggan_if,coggan_tss
392,2012-01-04 15:35:51+00:00,---------------,NaN,NaN,NaN,40.0
393,2012-01-05 12:32:51+00:00,---------------,NaN,NaN,NaN,80.0
394,2012-01-06 13:22:53+00:00,---------------,NaN,NaN,NaN,40.0
400,2012-04-01 14:30:30+00:00,TDS--C-AGL-----,NaN,NaN,NaN,100.0
420,2012-07-15 15:56:23+00:00,TDS--C-AGL-----,NaN,NaN,NaN,130.0
421,2012-07-20 10:54:12+00:00,TDS--C-AGL-----,NaN,NaN,NaN,75.0
422,2012-07-22 11:31:12+00:00,TDS--C-AGL-----,NaN,NaN,NaN,115.0
427,2012-07-28 09:24:16+00:00,TDS--C-AGL-----,NaN,NaN,0.94,88.0
431,2012-08-04 11:55:47+00:00,TDS--C-AGL-----,NaN,NaN,NaN,130.0
432,2012-08-05 13:27:44+00:00,TDS--C-AGL-----,NaN,NaN,NaN,75.0


### TSS 품질 표시 결과

파워 센서 없이 TSS가 존재하는 기록은 21개였다. 이 기록들은 모두
평균 파워와 NP가 비어 있었으며, IF는 1개 기록에만 존재했다.

TSS는 대부분 둥근 값으로 저장되어 있어 수동 입력, 외부에서 가져온
요약값 또는 추정값일 가능성이 있다. 장기 훈련 부하 분석에는 참고할
수 있지만, 파워 기반 분석에서는 별도로 구분한다.

## 11. 주요 지표의 범위 확인

주요 운동 지표의 최솟값, 중앙값, 최댓값 등을 확인하여 데이터의
전체적인 분포를 파악하고 이상치 후보를 찾는다.

이 단계에서는 값을 삭제하거나 수정하지 않고, 추가 확인이 필요한
기록을 찾는 데 집중한다.

In [24]:
range_check_columns = [
    "workout_time",
    "total_distance",
    "average_power",
    "max_power",
    "average_hr",
    "max_heartrate",
    "coggan_if",
    "coggan_tss",
]

range_summary_df = (
    cleaned_df[range_check_columns]
    .describe()
    .T
    .round(2)
)

range_summary_df

,count,mean,std,min,25%,50%,75%,max
workout_time,591.0,6225.83,4160.61,126.00,4260.00,5402.00,6757.93,40169.00
total_distance,563.0,42.87,26.36,0.00,29.37,39.70,49.81,197.18
average_power,469.0,171.85,36.10,37.48,147.91,169.55,196.51,274.39
max_power,469.0,588.92,117.11,129.00,515.00,591.00,661.00,885.00
average_hr,468.0,136.24,16.37,14.00,129.00,138.22,146.32,175.40
max_heartrate,465.0,167.93,17.97,14.00,162.00,171.00,177.00,229.00
coggan_if,470.0,0.84,0.12,0.24,0.78,0.86,0.92,1.13
coggan_tss,490.0,118.03,69.09,1.42,76.51,110.53,138.96,674.50


### 지표 범위 확인 결과

평균 심박과 최대 심박의 최솟값이 모두 14 bpm으로 나타났다.
라이딩 중 정상적인 심박수로 보기 어려우므로 센서 오류 또는 기록
문제일 가능성이 높다.

최대 심박의 최댓값인 229 bpm도 센서 오류 가능성이 있어 해당
라이드를 별도로 확인할 필요가 있다.

IF는 0.24부터 1.13, TSS는 1.42부터 674.50까지 넓은 범위를
보였다. 이러한 값은 운동 시간과 강도에 따라 실제로 나타날 수도
있으므로 값 하나만으로 이상치라고 판단하지 않는다.

다음 분석에서는 의심스러운 심박 기록을 직접 확인하고, IF와 TSS를
운동 시간 및 파워 지표와 함께 비교한다. 현재 단계에서는 어떤 값도
삭제하거나 수정하지 않는다.

## 12. 심박 이상치 후보 확인

요약 통계에서 발견한 비정상적으로 낮거나 높은 심박 기록을
직접 확인한다.

넓은 범위의 검토 조건을 사용하여 후보를 찾으며, 현재 단계에서는
해당 기록을 삭제하거나 실제 오류로 확정하지 않는다.

In [25]:
hr_range_needs_review = (
    (cleaned_df["average_hr"] < 40)
    | (cleaned_df["max_heartrate"] < 40)
    | (cleaned_df["max_heartrate"] > 220)
    | (cleaned_df["average_hr"] > cleaned_df["max_heartrate"])
)

print(
    "심박 검토 대상:",
    hr_range_needs_review.sum()
)

cleaned_df.loc[
    hr_range_needs_review,
    [
        "date",
        "data",
        "workout_time",
        "average_hr",
        "max_heartrate",
    ]
]

심박 검토 대상: 5


,date,data,workout_time,average_hr,max_heartrate
12,2005-08-28 23:06:00+00:00,TDS-H--A-L-----,1800.0,14.00000,14.0
14,2005-09-11 22:40:00+00:00,TDS-H--A-L-E---,6120.0,14.00000,14.0
42,2006-10-28 14:13:00+00:00,TDS-H--A-L-----,6360.0,26.00000,70.0
102,2007-06-04 17:56:03+00:00,T---H--A-------,5065.0,136.38401,228.0
317,2009-07-16 16:01:07+00:00,TDSPHC-AGL-----,6246.0,141.29711,229.0


In [26]:
cleaned_df["hr_range_needs_review"] = hr_range_needs_review

### 심박 이상치 후보 확인 결과

심박 범위 검토 대상으로 5개 라이드가 확인되었다. 모든 기록에는
심박 센서 표시가 존재했다.

- 평균·최대 심박이 모두 14 bpm인 기록: 2개
- 평균 심박이 26 bpm인 기록: 1개
- 최대 심박이 228 bpm 또는 229 bpm인 기록: 2개

14 bpm과 26 bpm은 라이딩 중 정상적인 평균 심박으로 보기 어려워
센서 또는 기록 오류일 가능성이 높다.

최대 심박이 높은 두 라이드는 평균 심박은 정상적인 범위였다.
원본 시계열을 확인한 결과 운동 초반에 높은 심박이 일정 시간 이어져,
짧은 구간의 센서 측정 오류일 가능성이 있다.

해당 라이드를 삭제하지 않고 `hr_range_needs_review` 열에 표시한다.
이후 심박 분석에서는 평균 심박과 최대 심박의 품질을 구분해서
판단할 필요가 있다.

## 13. IF와 TSS 계산 검증

`coggan_if` 리스트의 두 번째 요소를 별도로 추출하여
TSS 계산에 사용된 시간 또는 가중치 역할을 하는지 확인한다.

In [27]:
def extract_metric_weight(value):
    if isinstance(value, list) and len(value) > 1:
        return value[1]

    return None


cleaned_df["coggan_if_weight"] = pd.to_numeric(
    analysis_df["coggan_if"].map(
        extract_metric_weight
    ),
    errors="coerce",
)

cleaned_df[
   [
        "workout_time",
        "time_riding",
        "coggan_if",
        "coggan_if_weight",
        "coggan_tss",
    ]
].dropna().head()

,workout_time,time_riding,coggan_if,coggan_if_weight,coggan_tss
58,16138.16,16075.08,0.86544,16138.08,335.75887
76,15027.26,14939.82,0.91280,15026.76,347.78921
77,23940.26,23790.06,0.82586,23940.00,453.56478
79,3549.00,2816.00,1.01068,2817.00,79.93059
81,3549.42,3548.16,1.02252,3549.42,103.08512


In [28]:
# 계산 TSS = coggan_if_weight ÷ 3600 × coggan_if² × 100

cleaned_df["tss_from_formula"] = (
    cleaned_df["coggan_if_weight"]
    / 3600
    * cleaned_df["coggan_if"] ** 2
    * 100
)

cleaned_df["tss_absolute_difference"] = (
    cleaned_df["coggan_tss"]
    - cleaned_df["tss_from_formula"]
).abs()

In [29]:
power_tss_validation_mask = (
    cleaned_df["has_power_sensor"]
    & cleaned_df["coggan_tss"].notna()
    & cleaned_df["tss_from_formula"].notna()
)

cleaned_df.loc[
    power_tss_validation_mask,
    "tss_absolute_difference",
].describe().round(4)

count    469.0000
mean       0.0007
std        0.0007
min        0.0000
25%        0.0003
50%        0.0006
75%        0.0010
max        0.0058
Name: tss_absolute_difference, dtype: float64

### IF와 TSS 계산 검증 결과

`coggan_if` 리스트의 두 번째 요소는 라이드에 따라 `workout_time` 또는
`time_riding`에 가까운 값을 보였다. 따라서 특정 시간 열을 그대로 복사한
값이라기보다 IF와 TSS 계산에 실제로 사용된 시간 가중치로 해석하고
`coggan_if_weight` 열에 보존한다.

`coggan_if_weight / 3600 * coggan_if² * 100`으로 TSS를 다시 계산한 뒤,
파워 센서가 있는 469개 라이드의 저장된 TSS와 비교했다.

- 평균 절대 오차: 0.0007
- 중앙 절대 오차: 0.0006
- 최대 절대 오차: 0.0058

오차가 매우 작아 소수점 저장 과정의 반올림 차이로 볼 수 있다. 따라서
파워 센서가 있는 라이드의 TSS는 IF와 계산 가중 시간을 이용한 공식과
일관되게 계산된 것으로 판단한다.

파워 센서 없이 TSS가 존재하는 21개 기록은 계산에 필요한 값이 부족하므로
이번 공식 검증 대상에서 제외하고 별도의 품질 표시를 유지한다.

In [30]:
cleaned_df["if_from_formula"] = (
    cleaned_df["coggan_np"]
    / cleaned_df["cp_setting"]
)

cleaned_df["if_absolute_difference"] = (
    cleaned_df["coggan_if"]
    - cleaned_df["if_from_formula"]
).abs()

cleaned_df[
    "if_absolute_difference"
].dropna().describe().round(6)

count    469.000000
mean       0.012576
std        0.026495
min        0.000000
25%        0.000002
50%        0.000003
75%        0.000005
max        0.089970
Name: if_absolute_difference, dtype: float64

In [31]:
if_difference_needs_review = (
    cleaned_df["if_absolute_difference"] > 0.001
)

print(
    "IF 기준 불일치:",
    if_difference_needs_review.sum()
)

IF 기준 불일치: 90


In [32]:
cleaned_df["reference_power_from_stored_if"] = (
    cleaned_df["coggan_np"]
    / cleaned_df["coggan_if"]
)

cleaned_df.loc[
    if_difference_needs_review,
    [
        "date",
        "coggan_np",
        "cp_setting",
        "reference_power_from_stored_if",
        "coggan_if",
        "if_from_formula",
        "if_absolute_difference",
    ]
].nlargest(
    10,
    "if_absolute_difference"
)

,date,coggan_np,cp_setting,reference_power_from_stored_if,coggan_if,if_from_formula,if_absolute_difference
566,2015-08-09 09:59:09+00:00,202.42348,225.0,250.001210,0.80969,0.899660,0.089970
567,2015-08-11 11:08:50+00:00,201.04297,225.0,250.000584,0.80417,0.893524,0.089354
570,2015-08-28 14:30:59+00:00,192.92059,225.0,250.000765,0.77168,0.857425,0.085745
569,2015-08-16 12:38:21+00:00,191.20816,225.0,250.000863,0.76483,0.849814,0.084984
571,2015-09-06 11:50:38+00:00,189.15605,225.0,250.001388,0.75662,0.840694,0.084074
380,2010-10-13 17:40:31+00:00,241.48951,255.0,235.001129,1.02761,0.947018,0.080592
580,2016-08-10 18:21:42+00:00,179.04927,225.0,249.998981,0.71620,0.795775,0.079575
384,2010-10-20 16:55:43+00:00,238.21560,255.0,235.000789,1.01368,0.934179,0.079501
369,2010-09-22 16:11:33+00:00,234.37438,255.0,234.999479,0.99734,0.919115,0.078225
575,2016-05-05 11:57:18+00:00,173.68722,225.0,249.999597,0.69475,0.771943,0.077193


In [33]:
cleaned_df["reference_power_rounded"] = (
    cleaned_df["reference_power_from_stored_if"].round()
)

reference_power_patterns = (
    cleaned_df.loc[
        if_difference_needs_review,
        [
            "cp_setting",
            "reference_power_rounded",
        ]
    ]
    .value_counts()
)

reference_power_patterns

cp_setting  reference_power_rounded
255.0       235.0                      64
225.0       250.0                      22
280.0       275.0                       2
281.0       275.0                       1
239.0       235.0                       1
Name: count, dtype: int64

In [34]:
reference_power_periods = (
    cleaned_df.loc[
        if_difference_needs_review,
        [
            "date",
            "cp_setting",
            "reference_power_rounded",
        ],
    ]
    .groupby(
        [
            "cp_setting",
            "reference_power_rounded",
        ]
    )["date"]
    .agg(["count", "min", "max"])
)

reference_power_periods

,,count,min,max
cp_setting,reference_power_rounded,,,
225.0,250.0,22,2015-08-07 13:37:54+00:00,2016-08-29 10:14:46+00:00
239.0,235.0,1,2010-10-11 11:35:49+00:00,2010-10-11 11:35:49+00:00
255.0,235.0,64,2010-09-01 17:15:34+00:00,2012-07-30 17:48:22+00:00
280.0,275.0,2,2007-05-13 05:20:15+00:00,2007-07-03 12:18:18+00:00
281.0,275.0,1,2009-05-13 18:18:39+00:00,2009-05-13 18:18:39+00:00


In [35]:
reference_power_period_mask = (
    cleaned_df["date"].between(
        "2010-09-01",
        "2012-07-30 23:59:59"
    )
    & cleaned_df["has_power_sensor"]
)

print("기간 내 파워 라이드:")
print(reference_power_period_mask.sum())

print("\n역산 기준 파워:")
print(
    cleaned_df.loc[
        reference_power_period_mask,
        "reference_power_rounded",
    ]
    .value_counts()
)

print("\n저장된 cp_setting:")
print(
    cleaned_df.loc[
        reference_power_period_mask,
        "cp_setting",
    ]
    .value_counts()
)

기간 내 파워 라이드:
65

역산 기준 파워:
reference_power_rounded
235.0    65
Name: count, dtype: int64

저장된 cp_setting:
cp_setting
255.0    64
239.0     1
Name: count, dtype: int64


### IF 계산 검증 결과

`coggan_np / cp_setting`으로 IF를 다시 계산하여 저장된 `coggan_if`와
비교했다. NP가 존재하는 469개 라이드 중 대부분은 두 값이 소수점
반올림 수준에서 일치했지만, 절대 오차가 0.001보다 큰 기록이 90개
확인되었다. 최대 절대 오차는 약 0.09로 반올림만으로 설명할 수 없다.

저장된 IF에서 `coggan_np / coggan_if`로 기준 파워를 역산한 결과,
불일치 기록은 몇 가지 반복 패턴으로 묶였다.

- `cp_setting` 255W, 역산 기준 235W: 64개
- `cp_setting` 225W, 역산 기준 250W: 22개
- `cp_setting` 280W, 역산 기준 275W: 2개
- 그 밖의 조합: 2개

특히 2010년 9월 1일부터 2012년 7월 30일까지 파워 센서가 있는
65개 라이드 모두 IF 역산 기준이 235W였다. 이 중 64개는
`cp_setting`이 255W이고 1개는 239W였다. 같은 기준이 특정 기간에
연속적으로 사용되었으므로 개별 라이드의 우연한 계산 오류보다는
별도의 FTP 또는 Coggan 계산 기준이 설정되어 있었을 가능성이 높다.

따라서 `cp_setting`이 항상 IF 계산의 분모라고 가정하지 않는다. 원본
`cp_setting`과 역산한 기준 파워를 모두 보존하며, 역산값은 저장된 IF를
설명하기 위한 파생값으로만 사용한다. 역산값은 NP와 IF에서 만들어진
값이므로 실제 체력 기준의 독립적인 정답이나 머신러닝 입력값으로
그대로 사용하지 않는다.

In [36]:
cleaned_df["workout_hours"] = (
        cleaned_df["workout_time"]
        / 3600
)

if_context_columns = [
    "date",
    "workout_hours",
    "average_power",
    "coggan_np",
    "cp_setting",
    "reference_power_rounded",
    "coggan_if",
    "coggan_tss",
]

print("IF가 높은 라이드")

display(
    cleaned_df.nlargest(
        5,
        "coggan_if",
    )[if_context_columns]
)

print("IF가 낮은 라이드")

display(
    cleaned_df.nsmallest(
        5,
        "coggan_if",
    )[if_context_columns]
)

IF가 높은 라이드


,date,workout_hours,average_power,coggan_np,cp_setting,reference_power_rounded,coggan_if,coggan_tss
130,2008-07-22 16:44:44+00:00,0.330278,250.70227,282.12618,250.0,250.0,1.12850,42.06163
128,2008-07-16 17:57:26+00:00,0.325556,248.56143,281.26070,250.0,250.0,1.12504,41.20626
352,2010-08-27 15:41:41+00:00,1.361389,200.04754,223.36018,200.0,200.0,1.11680,169.79845
517,2014-11-21 17:19:42+00:00,0.163056,217.00511,239.78078,220.0,220.0,1.08991,19.36953
180,2008-12-21 14:49:35+00:00,1.296944,242.25294,264.14502,250.0,250.0,1.05658,144.78588


IF가 낮은 라이드


,date,workout_hours,average_power,coggan_np,cp_setting,reference_power_rounded,coggan_if,coggan_tss
228,2009-03-15 11:00:04+00:00,0.390278,37.47568,67.00193,275.0,275.0,0.24364,2.30523
126,2008-05-18 21:38:31+00:00,0.086398,101.00965,101.25792,250.0,250.0,0.40503,1.41722
245,2009-04-06 09:14:47+00:00,0.848333,102.35828,114.89606,275.0,275.0,0.41780,14.64365
578,2016-08-07 10:34:21+00:00,0.909722,55.48748,115.54874,225.0,250.0,0.46219,19.43387
115,2007-11-19 18:43:45+00:00,1.001556,146.08800,147.81254,275.0,275.0,0.53750,29.55012


### IF 극단값 확인 결과

IF가 높은 라이드와 낮은 라이드를 운동 시간, 파워, 기준 파워 및
TSS와 함께 확인했다.

높은 IF 상위 기록은 대부분 비교적 짧은 고강도 라이드였으며, 저장된
`cp_setting`과 IF에서 역산한 기준 파워도 일치했다. 따라서 기준 파워
불일치 때문에 IF가 비정상적으로 높아진 것으로 보이지 않는다. 다만
약 1시간 이상 지속된 높은 IF 기록은 당시 기준 파워가 실제 체력보다
낮게 설정되었거나 경기·테스트였을 가능성을 추가로 고려할 수 있다.

낮은 IF 상위 기록도 짧거나 평균 파워와 NP가 낮은 라이드였다. 578번
라이드는 `cp_setting` 225W와 역산 기준 250W가 달랐지만, 225W로 다시
계산해도 IF는 약 0.51로 낮다. 따라서 기준 파워 차이가 저장된 IF를 더
낮추기는 했지만 낮은 강도의 주된 원인은 아니다.

최저 IF 라이드는 최대 파워가 497W였지만 평균 파워 37W, NP 67W로
대부분의 시간에는 매우 가볍게 이동한 것으로 보인다. 최대 파워가
잠시 높더라도 운동 전체의 NP가 낮으면 IF는 낮게 계산될 수 있다.

현재 IF 범위 0.24–1.13은 운동 시간과 전체 강도로 설명할 수 있으므로
범위만을 이유로 값을 삭제하거나 수정하지 않는다. 다만 파워 센서와
NP 없이 IF만 존재하는 기록 1개는 계산을 검증할 수 없으므로 기존의
품질 구분을 유지한다.

In [37]:
tss_context_columns = [
    "date",
    "workout_hours",
    "average_power",
    "coggan_np",
    "coggan_if",
    "coggan_tss",
    "has_power_sensor",
    "tss_without_power_sensor",
]

print("TSS가 높은 라이드")

display(
    cleaned_df.nlargest(
        5,
        "coggan_tss",
    )[tss_context_columns]
)

print("TSS가 낮은 라이드")

display(
    cleaned_df.nsmallest(
        5,
        "coggan_tss",
    )[tss_context_columns]
)

TSS가 높은 라이드


,date,workout_hours,average_power,coggan_np,coggan_if,coggan_tss,has_power_sensor,tss_without_power_sensor
96,2007-05-13 05:20:15+00:00,8.409722,190.12172,246.28281,0.89557,674.50390,True,False
97,2007-05-19 23:00:00+00:00,7.992794,172.10396,224.28974,0.81560,509.27185,True,False
93,2007-04-21 23:00:00+00:00,6.452711,193.11846,235.79961,0.85745,474.41185,True,False
313,2009-07-04 05:09:48+00:00,11.158056,132.64348,176.41322,0.64150,459.14827,True,False
77,2007-04-07 23:00:00+00:00,6.650072,179.75574,227.11276,0.82586,453.56478,True,False


TSS가 낮은 라이드


,date,workout_hours,average_power,coggan_np,coggan_if,coggan_tss,has_power_sensor,tss_without_power_sensor
126,2008-05-18 21:38:31+00:00,0.086398,101.00965,101.25792,0.40503,1.41722,True,False
562,2015-07-21 11:19:33+00:00,0.035000,126.90476,145.68628,0.66221,1.53483,True,False
228,2009-03-15 11:00:04+00:00,0.390278,37.47568,67.00193,0.24364,2.30523,True,False
214,2009-02-21 09:47:33+00:00,0.225833,127.31527,158.25539,0.57547,7.46973,True,False
194,2009-01-16 15:37:26+00:00,0.250278,181.26699,196.31860,0.74082,12.56192,True,False


In [38]:
cleaned_df["reference_power_differs_from_cp"] = (
    if_difference_needs_review
)

cleaned_df["tss_using_cp_setting"] = (
    cleaned_df["coggan_if_weight"]
    / 3600
    * cleaned_df["if_from_formula"] ** 2
    * 100
)

cleaned_df["tss_vs_cp_percent_difference"] = (
    (
        cleaned_df["coggan_tss"]
        - cleaned_df["tss_using_cp_setting"]
    )
    / cleaned_df["tss_using_cp_setting"]
    * 100
)

cleaned_df.loc[
    cleaned_df["reference_power_differs_from_cp"],
    [
        "date",
        "cp_setting",
        "reference_power_rounded",
        "coggan_tss",
        "tss_using_cp_setting",
        "tss_vs_cp_percent_difference",
    ],
].head()

,date,cp_setting,reference_power_rounded,coggan_tss,tss_using_cp_setting,tss_vs_cp_percent_difference
96,2007-05-13 05:20:15+00:00,280.0,275.0,674.50390,650.629557,3.669422
107,2007-07-03 12:18:18+00:00,280.0,275.0,90.85627,87.640381,3.669415
273,2009-05-13 18:18:39+00:00,281.0,275.0,130.17876,124.678871,4.411244
356,2010-09-01 17:15:34+00:00,255.0,235.0,109.17243,92.718920,17.745580
357,2010-09-03 16:11:30+00:00,255.0,235.0,115.64436,98.215454,17.745584


### TSS 극단값 확인 결과

TSS가 높은 라이드와 낮은 라이드를 운동 시간과 IF를 함께 비교했다.
TSS 상위 5개 기록은 모두 장시간 라이드였으며, IF도 약 0.64–0.90으로
나타났다. 313번 라이드는 운동 시간이 약 11.2시간으로 가장 길지만
IF가 약 0.64로 비교적 낮아 TSS는 네 번째로 높았다. 이는 TSS가 운동
시간뿐 아니라 IF의 제곱에도 영향을 받는다는 계산 구조와 일치한다.

TSS 하위 기록은 운동 시간이 매우 짧거나 IF가 매우 낮았다. 562번
라이드는 IF가 약 0.66이지만 운동 시간이 약 2.1분에 불과하여 낮은
TSS가 자연스럽다. 반대로 228번 라이드는 약 23.4분 동안 기록되었지만
IF가 약 0.24로 매우 낮아 TSS가 작게 계산되었다.

상위 및 하위 5개 기록에는 모두 파워 센서가 포함되어 있었고, 운동
시간과 IF를 함께 고려했을 때 극단적인 TSS도 설명 가능했다. 따라서
현재 확인한 TSS 극단값은 범위만을 이유로 삭제하거나 수정하지 않는다.
다만 TSS는 기준 파워 설정의 영향을 받으므로, IF 계산 기준이
`cp_setting`과 다른 기록은 기존 품질 표시를 유지한다.

## 14. 시간 지표 검증

`workout_time`과 `time_riding`의 결측값, 값의 범위 및 두 시간의
관계를 확인한다.

In [39]:
riding_time_longer_than_workout = (
    cleaned_df["time_riding"]
    > cleaned_df["workout_time"]
)

print(
    "라이드 타임이 전체 운동 시간보다 긴 기록:",
    riding_time_longer_than_workout.sum()
)

cleaned_df.loc[
    riding_time_longer_than_workout,
    [
        "date",
        "workout_time",
        "time_riding",
    ]
]

라이드 타임이 전체 운동 시간보다 긴 기록: 1


,date,workout_time,time_riding
115,2007-11-19 18:43:45+00:00,3605.602,3657.408


In [40]:
cleaned_df["riding_time_longer_than_workout"] = (
    riding_time_longer_than_workout
)

In [41]:
workout_time_not_positive = (
    cleaned_df["workout_time"] <= 0
)

riding_time_not_positive = (
    cleaned_df["time_riding"] <= 0
)

print(
    "전체 운동 시간이 0 이하인 기록:",
    workout_time_not_positive.sum()
)

print(
    "라이드 시간이 0 이하인 기록:",
    riding_time_not_positive.sum()
)

전체 운동 시간이 0 이하인 기록: 0
라이드 시간이 0 이하인 기록: 0


In [42]:
workout_time_missing = (
    cleaned_df["workout_time"].isna()
)

riding_time_missing = (
    cleaned_df["time_riding"].isna()
)

print(
    "전체 운동 시간이 비어 있는 기록:",
    workout_time_missing.sum()
)

print(
    "라이드 시간이 비어 있는 기록:",
    riding_time_missing.sum()
)

time_missing = (
    workout_time_missing
    | riding_time_missing
)

cleaned_df.loc[
    time_missing,
    [
        "date",
        "workout_time",
        "time_riding",
        "total_distance",
    ]
]

전체 운동 시간이 비어 있는 기록: 1
라이드 시간이 비어 있는 기록: 18


,date,workout_time,time_riding,total_distance
7,2005-07-18 18:31:00+00:00,1440.0,NaN,NaN
21,2006-06-02 22:31:00+00:00,8760.0,NaN,NaN
22,2006-06-04 16:24:00+00:00,4920.0,NaN,NaN
43,2006-11-07 20:00:46+00:00,1635.0,NaN,NaN
47,2006-12-01 17:06:41+00:00,4310.0,NaN,NaN
54,2007-01-22 10:10:13+00:00,7660.0,NaN,NaN
55,2007-01-23 08:23:57+00:00,3715.0,NaN,NaN
60,2007-02-12 20:00:06+00:00,3375.0,NaN,NaN
61,2007-02-13 06:47:52+00:00,3460.0,NaN,NaN
62,2007-02-14 13:13:26+00:00,2410.0,NaN,NaN


In [43]:
cleaned_df["workout_time_missing"] = (
    workout_time_missing
)

cleaned_df["riding_time_missing"] = (
    riding_time_missing
)

In [44]:
cleaned_df["stopped_time_minutes"] = (
    cleaned_df["workout_time"]
    - cleaned_df["time_riding"]
) / 60

cleaned_df["riding_time_ratio"] = (
    cleaned_df["time_riding"]
    / cleaned_df["workout_time"]
)

cleaned_df[
    [
        "stopped_time_minutes",
        "riding_time_ratio",
    ]
].describe()

,stopped_time_minutes,riding_time_ratio
count,574.000000,574.000000
mean,3.512810,0.971714
std,13.010853,0.080919
min,-0.863433,0.000687
25%,0.237500,0.973727
50%,0.766667,0.991620
75%,2.645833,0.997112
max,242.416667,1.014368


In [45]:
time_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "stopped_time_minutes",
    "riding_time_ratio",
    "total_distance",
    "average_speed",
]

print("정지 시간이 가장 긴 기록")

display(
    cleaned_df.nlargest(
        5,
        "stopped_time_minutes",
    )[time_context_columns]
)

print("실제 라이딩 시간 비율이 가장 낮은 기록")

display(
    cleaned_df.nsmallest(
        5,
        "riding_time_ratio",
    )[time_context_columns]
)

정지 시간이 가장 긴 기록


,date,workout_time,time_riding,stopped_time_minutes,riding_time_ratio,total_distance,average_speed
53,2007-01-21 09:23:24+00:00,14555.00,10.0,242.416667,0.000687,NaN,NaN
17,2005-11-21 00:48:00+00:00,7200.00,1060.0,102.333333,0.147222,5.080,17.25283
56,2007-01-23 09:26:08+00:00,5470.00,50.0,90.333333,0.009141,0.002,1.44000
313,2009-07-04 05:09:48+00:00,40169.00,35779.0,73.166667,0.890712,175.692,17.74418
120,2007-11-25 09:04:29+00:00,18540.86,14660.1,64.679333,0.790691,97.550,24.22760


실제 라이딩 시간 비율이 가장 낮은 기록


,date,workout_time,time_riding,stopped_time_minutes,riding_time_ratio,total_distance,average_speed
53,2007-01-21 09:23:24+00:00,14555.0,10.0,242.416667,0.000687,NaN,NaN
56,2007-01-23 09:26:08+00:00,5470.0,50.0,90.333333,0.009141,0.00200,1.44000
17,2005-11-21 00:48:00+00:00,7200.0,1060.0,102.333333,0.147222,5.08000,17.25283
49,2006-12-08 09:51:52+00:00,2395.0,815.0,26.333333,0.340292,0.93786,4.19416
578,2016-08-07 10:34:21+00:00,3275.0,2369.0,15.100000,0.723359,12.37944,18.82010


In [46]:
cleaned_df["time_relationship_needs_review"] = (
    riding_time_longer_than_workout
    | (cleaned_df["riding_time_ratio"] < 0.05)
)

cleaned_df.loc[
    cleaned_df["time_relationship_needs_review"],
    time_context_columns,
]

,date,workout_time,time_riding,stopped_time_minutes,riding_time_ratio,total_distance,average_speed
53,2007-01-21 09:23:24+00:00,14555.000,10.000,242.416667,0.000687,NaN,NaN
56,2007-01-23 09:26:08+00:00,5470.000,50.000,90.333333,0.009141,0.002,1.44000
115,2007-11-19 18:43:45+00:00,3605.602,3657.408,-0.863433,1.014368,31.941,31.58223


### 시간 지표 검증 결과

시간이 0 이하인 기록은 없었다. `time_riding`은 18개 기록에서
비어 있었으며, 이 중 403번 기록은 `workout_time`도 함께 비어
있었다. 결측값은 0과 의미가 다르므로 채우지 않고
`workout_time_missing`과 `riding_time_missing`으로 구분한다.

전체 시간 중 실제 라이딩 시간의 비율을 확인한 결과, 대부분의
기록은 두 시간이 서로 비슷했다. 정지 시간이 길더라도 313번과
120번처럼 장거리 라이딩의 휴식으로 설명할 수 있는 기록은 유지한다.
17번 기록도 실제 라이딩 시간, 거리 및 평균 속도가 서로 일관되므로
자동으로 제외하지 않는다.

실제 라이딩 시간 비율이 5% 미만인 53번과 56번, `time_riding`이
`workout_time`보다 약 52초 긴 115번을
`time_relationship_needs_review`로 표시했다. 이 세 기록은 원본 값을
수정하거나 삭제하지 않고, 시간 기반 분석에서 별도로 검토한다.

## 15. 거리와 평균 속도 검증

`total_distance`, `time_riding`, `average_speed`의 관계와 결측값 및
극단값을 확인한다.

In [47]:
cleaned_df["average_speed_from_distance_and_riding_time"] = (
    cleaned_df["total_distance"]
    / (cleaned_df["time_riding"] / 3600)
)

cleaned_df["average_speed_riding_time_absolute_difference"] = (
    cleaned_df["average_speed"]
    - cleaned_df["average_speed_from_distance_and_riding_time"]
).abs()

cleaned_df["average_speed_from_distance_and_workout_time"] = (
    cleaned_df["total_distance"]
    / (cleaned_df["workout_time"] / 3600)
)

cleaned_df["average_speed_workout_time_absolute_difference"] = (
    cleaned_df["average_speed"]
    - cleaned_df["average_speed_from_distance_and_workout_time"]
).abs()

cleaned_df[
    [
        "average_speed_riding_time_absolute_difference",
        "average_speed_workout_time_absolute_difference",
    ]
].describe()

,average_speed_riding_time_absolute_difference,average_speed_workout_time_absolute_difference
count,5.630000e+02,5.630000e+02
mean,1.093988e-01,6.527514e-01
std,5.713044e-01,1.198869e+00
min,3.541913e-08,8.500773e-07
25%,3.974449e-06,1.005235e-01
50%,1.124252e-02,2.519761e-01
75%,3.273559e-02,7.041810e-01
max,9.191827e+00,1.471283e+01


In [48]:
speed_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "total_distance",
    "average_speed",
    "average_speed_from_distance_and_riding_time",
    "average_speed_riding_time_absolute_difference",
    "time_relationship_needs_review",
]

display(
    cleaned_df.nlargest(
        10,
        "average_speed_riding_time_absolute_difference",
    )[speed_context_columns]
)

,date,workout_time,time_riding,total_distance,average_speed,average_speed_from_distance_and_riding_time,average_speed_riding_time_absolute_difference,time_relationship_needs_review
65,2007-02-23 06:56:21+00:00,6395.00,5515.00,7.31222,13.96499,4.773163,9.191827,False
74,2007-03-18 21:02:30+00:00,1505.00,1495.00,0.00700,5.04000,0.016856,5.023144,False
30,2006-07-15 17:54:00+00:00,4680.00,4500.00,8.39000,11.61692,6.712000,4.904920,False
29,2006-07-14 17:42:00+00:00,5160.00,4560.00,7.98950,10.81286,6.307500,4.505360,False
32,2006-08-15 12:42:00+00:00,2040.00,1960.00,18.87850,38.61511,34.674796,3.940314,False
46,2006-11-18 14:01:37+00:00,9015.00,8215.00,0.00300,2.16000,0.001315,2.158685,False
95,2007-05-11 10:56:02+00:00,4915.00,4680.00,0.00200,1.44000,0.001538,1.438462,False
56,2007-01-23 09:26:08+00:00,5470.00,50.00,0.00200,1.44000,0.144000,1.296000,True
132,2008-08-16 12:00:26+00:00,14329.76,12891.06,77.41200,22.90567,21.618331,1.287339,False
96,2007-05-13 05:20:15+00:00,30275.00,30090.00,175.16700,22.06056,20.957168,1.103392,False


In [49]:
cleaned_df["speed_relationship_needs_review"] = (
    cleaned_df[
        "average_speed_riding_time_absolute_difference"
    ] > 0.5
)

print(
    "속도 관계 검토가 필요한 기록:",
    cleaned_df["speed_relationship_needs_review"].sum()
)

속도 관계 검토가 필요한 기록: 22


In [50]:
speed_review_dates = cleaned_df.loc[
    cleaned_df["speed_relationship_needs_review"],
    "date",
]

print(
    "가장 이른 날짜:",
    speed_review_dates.min()
)

print(
    "가장 늦은 날짜:",
    speed_review_dates.max()
)

speed_review_dates.dt.year.value_counts().sort_index()

가장 이른 날짜: 2006-07-14 17:42:00+00:00
가장 늦은 날짜: 2010-10-23 16:36:08+00:00


date
2006    5
2007    5
2008    9
2009    2
2010    1
Name: count, dtype: int64

In [51]:
speed_and_time_need_review = (
    cleaned_df["speed_relationship_needs_review"]
    & cleaned_df["time_relationship_needs_review"]
)

print(
    "시간과 속도 모두 검토가 필요한 기록:",
    speed_and_time_need_review.sum()
)

시간과 속도 모두 검토가 필요한 기록: 1


In [52]:
distance_not_positive = (
    cleaned_df["total_distance"] <= 0
)

average_speed_not_positive = (
    cleaned_df["average_speed"] <= 0
)

print(
    "거리가 0 이하인 기록:",
    distance_not_positive.sum()
)

print(
    "평균 속도가 0 이하인 기록:",
    average_speed_not_positive.sum()
)

print(
    "거리가 비어 있는 기록:",
    cleaned_df["total_distance"].isna().sum()
)

print(
    "평균 속도가 비어 있는 기록:",
    cleaned_df["average_speed"].isna().sum()
)

distance_missing = (
    cleaned_df["total_distance"].isna()
)

average_speed_missing = (
    cleaned_df["average_speed"].isna()
)

distance_speed_missing_mismatch = (
    distance_missing
    != average_speed_missing
)

print(
    "거리와 속도의 결측 상태가 다른 기록:",
    distance_speed_missing_mismatch.sum()
)

거리가 0 이하인 기록: 0
평균 속도가 0 이하인 기록: 0
거리가 비어 있는 기록: 29
평균 속도가 비어 있는 기록: 29
거리와 속도의 결측 상태가 다른 기록: 0


In [53]:
cleaned_df["distance_speed_missing"] = (
    distance_missing
    & average_speed_missing
)

In [54]:
distance_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "total_distance",
    "average_speed",
    "speed_relationship_needs_review",
]

print("거리가 가장 짧은 기록")

display(
    cleaned_df.nsmallest(
        10,
        "total_distance",
    )[distance_context_columns]
)

print("거리가 가장 긴 기록")

display(
    cleaned_df.nlargest(
        5,
        "total_distance",
    )[distance_context_columns]
)

거리가 가장 짧은 기록


,date,workout_time,time_riding,total_distance,average_speed,speed_relationship_needs_review
56,2007-01-23 09:26:08+00:00,5470.000,50.0,0.00200,1.44000,True
95,2007-05-11 10:56:02+00:00,4915.000,4680.0,0.00200,1.44000,True
46,2006-11-18 14:01:37+00:00,9015.000,8215.0,0.00300,2.16000,True
74,2007-03-18 21:02:30+00:00,1505.000,1495.0,0.00700,5.04000,True
562,2015-07-21 11:19:33+00:00,126.000,123.0,0.60574,17.72912,False
49,2006-12-08 09:51:52+00:00,2395.000,815.0,0.93786,4.19416,False
126,2008-05-18 21:38:31+00:00,311.031,309.0,3.46400,40.48831,False
228,2009-03-15 11:00:04+00:00,1405.000,1321.0,3.76000,10.24678,False
207,2009-02-14 13:53:09+00:00,1223.000,1087.0,5.05000,16.74033,False
17,2005-11-21 00:48:00+00:00,7200.000,1060.0,5.08000,17.25283,False


거리가 가장 긴 기록


,date,workout_time,time_riding,total_distance,average_speed,speed_relationship_needs_review
110,2007-07-16 05:07:09+00:00,36570.00,34725.0,197.17967,20.44195,False
97,2007-05-19 23:00:00+00:00,28774.06,27405.0,184.76871,24.38048,False
33,2006-08-20 09:11:00+00:00,24840.00,24200.0,183.79000,27.34066,False
98,2007-05-20 05:59:16+00:00,28815.00,27350.0,182.16000,23.97718,False
313,2009-07-04 05:09:48+00:00,40169.00,35779.0,175.69200,17.74418,False


In [55]:
speed_range_context_columns = [
    "date",
    "time_riding",
    "total_distance",
    "average_speed",
    "average_speed_from_distance_and_riding_time",
    "average_speed_riding_time_absolute_difference",
    "speed_relationship_needs_review",
]

print("평균 속도가 가장 낮은 기록")

display(
    cleaned_df.nsmallest(
        5,
        "average_speed",
    )[speed_range_context_columns]
)

print("평균 속도가 가장 높은 기록")

display(
    cleaned_df.nlargest(
        5,
        "average_speed",
    )[speed_range_context_columns]
)

평균 속도가 가장 낮은 기록


,date,time_riding,total_distance,average_speed,average_speed_from_distance_and_riding_time,average_speed_riding_time_absolute_difference,speed_relationship_needs_review
56,2007-01-23 09:26:08+00:00,50.0,0.00200,1.44000,0.144000,1.296000,True
95,2007-05-11 10:56:02+00:00,4680.0,0.00200,1.44000,0.001538,1.438462,True
46,2006-11-18 14:01:37+00:00,8215.0,0.00300,2.16000,0.001315,2.158685,True
49,2006-12-08 09:51:52+00:00,815.0,0.93786,4.19416,4.142694,0.051466,False
74,2007-03-18 21:02:30+00:00,1495.0,0.00700,5.04000,0.016856,5.023144,True


평균 속도가 가장 높은 기록


,date,time_riding,total_distance,average_speed,average_speed_from_distance_and_riding_time,average_speed_riding_time_absolute_difference,speed_relationship_needs_review
79,2007-04-12 23:00:00+00:00,2816.0,32.66508,41.90816,41.759335,0.148825,False
385,2010-10-23 16:36:08+00:00,3056.0,34.73720,41.60144,40.920785,0.680655,True
126,2008-05-18 21:38:31+00:00,309.0,3.46400,40.48831,40.357282,0.131028,False
32,2006-08-15 12:42:00+00:00,1960.0,18.87850,38.61511,34.674796,3.940314,True
113,2007-11-17 21:22:59+00:00,3595.0,38.33500,38.52764,38.388317,0.139323,False


### 거리와 평균 속도 검증 결과

거리와 평균 속도는 같은 29개 기록에서 함께 비어 있었고, 하나만
비어 있는 기록이나 0 이하의 값은 없었다. 두 값이 함께 결측인 기록은
`distance_speed_missing`으로 표시한다.

거리와 `time_riding`으로 평균 속도를 다시 계산한 결과, 저장된
`average_speed`와의 차이 중앙값은 약 0.011km/h였다. 전체 운동
시간을 사용했을 때의 중앙값 약 0.252km/h보다 작으므로, 저장된 평균
속도는 대체로 실제 라이딩 시간을 기준으로 계산된 것으로 보인다.

두 평균 속도의 차이가 0.5km/h를 넘는 22개 기록을
`speed_relationship_needs_review`로 표시했다. 이 기록들은 특정 연도에
집중되지 않았으며, 시간 관계 검토 기록과 겹친 것은 56번 한 개였다.
따라서 대부분의 속도 불일치는 시간 관계 이상만으로 설명되지 않는다.

매우 짧거나 평균 속도가 극단적인 기록 중에는 지표 관계가 어긋난
기록과, 49번처럼 느리지만 거리·시간·속도가 일관된 기록이 함께
있었다. 높은 평균 속도 기록도 같은 방식으로 유효한 기록과 검토
대상을 구분할 수 있었다. 따라서 거리나 속도의 크기만으로 값을
삭제하지 않고, 지표 사이의 관계를 품질 판단에 사용한다.

## 16. 파워 지표 검증

평균 파워, NP, 최대 파워의 관계와 극단값을 확인하고 파워 변동성을
나타내는 VI를 탐색한다.

In [56]:
average_power_above_max = (
    cleaned_df["average_power"]
    > cleaned_df["max_power"]
)

print(
    "평균 파워가 최대 파워보다 큰 기록:",
    average_power_above_max.sum()
)

np_above_max_power = (
    cleaned_df["coggan_np"]
    > cleaned_df["max_power"]
)

print(
    "NP가 최대 파워보다 큰 기록:",
    np_above_max_power.sum()
)

average_power_above_np = (
    cleaned_df["average_power"]
    > cleaned_df["coggan_np"]
)

print(
    "평균 파워가 NP보다 큰 기록:",
    average_power_above_np.sum()
)

cleaned_df.loc[
    average_power_above_np,
    [
        "date",
        "workout_time",
        "average_power",
        "coggan_np",
        "max_power",
    ]
]

평균 파워가 최대 파워보다 큰 기록: 0
NP가 최대 파워보다 큰 기록: 0
평균 파워가 NP보다 큰 기록: 4


,date,workout_time,average_power,coggan_np,max_power
202,2009-02-10 18:00:00+00:00,3000.0,200.0,199.60736,200.0
203,2009-02-11 18:00:00+00:00,3600.0,195.0,194.68114,195.0
204,2009-02-12 00:00:00+00:00,3600.0,195.0,194.68114,195.0
205,2009-02-12 12:00:00+00:00,3600.0,195.0,194.68114,195.0


In [57]:
power_range_context_columns = [
    "date",
    "workout_time",
    "average_power",
    "coggan_np",
    "max_power",
    "coggan_if",
    "coggan_tss",
    "has_power_sensor",
]

print("평균 파워가 가장 낮은 기록")

display(
    cleaned_df.nsmallest(
        5,
        "average_power",
    )[power_range_context_columns]
)

print("평균 파워가 가장 높은 기록")

display(
    cleaned_df.nlargest(
        5,
        "average_power",
    )[power_range_context_columns]
)

평균 파워가 가장 낮은 기록


,date,workout_time,average_power,coggan_np,max_power,coggan_if,coggan_tss,has_power_sensor
228,2009-03-15 11:00:04+00:00,1405.000,37.47568,67.00193,497.000,0.24364,2.30523,True
578,2016-08-07 10:34:21+00:00,3275.000,55.48748,115.54874,428.000,0.46219,19.43387,True
565,2015-08-07 13:37:54+00:00,10176.000,64.53270,152.86719,636.271,0.61147,105.34464,True
572,2016-04-13 10:54:38+00:00,4360.000,98.98658,164.72374,668.000,0.65889,52.57949,True
126,2008-05-18 21:38:31+00:00,311.031,101.00965,101.25792,129.000,0.40503,1.41722,True


평균 파워가 가장 높은 기록


,date,workout_time,average_power,coggan_np,max_power,coggan_if,coggan_tss,has_power_sensor
225,2009-03-09 12:35:23+00:00,3732.00,274.38613,288.85882,748.0,1.05040,113.58181,True
81,2007-04-13 10:00:00+00:00,3549.42,263.20518,281.19217,644.0,1.02252,103.08512,True
79,2007-04-12 23:00:00+00:00,3549.00,263.20483,277.93742,644.0,1.01068,79.93059,True
273,2009-05-13 18:18:39+00:00,4517.00,256.64667,280.11022,559.0,1.01858,130.17876,True
254,2009-04-20 10:33:16+00:00,4802.00,255.87262,276.03359,661.0,1.00376,133.80574,True


In [58]:
cleaned_df["variability_index"] = (
    cleaned_df["coggan_np"]
    / cleaned_df["average_power"]
)

variability_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "total_distance",
    "elevation_gain",
    "average_power",
    "coggan_np",
    "max_power",
    "variability_index",
]

print("VI가 가장 높은 기록")

display(
    cleaned_df.nlargest(
        5,
        "variability_index",
    )[variability_context_columns]
)

VI가 가장 높은 기록


,date,workout_time,time_riding,total_distance,elevation_gain,average_power,coggan_np,max_power,variability_index
565,2015-08-07 13:37:54+00:00,10176.0,7992.0,25.00450,745.8000,64.53270,152.86719,636.271,2.368833
578,2016-08-07 10:34:21+00:00,3275.0,2369.0,12.37944,150.7385,55.48748,115.54874,428.000,2.082429
228,2009-03-15 11:00:04+00:00,1405.0,1321.0,3.76000,12.6000,37.47568,67.00193,497.000,1.787878
566,2015-08-09 09:59:09+00:00,3492.0,2796.0,12.68110,371.1310,113.38508,202.42348,759.000,1.785274
572,2016-04-13 10:54:38+00:00,4360.0,3860.0,24.98528,200.0000,98.98658,164.72374,668.000,1.664102


In [59]:
max_power_context_columns = [
    "date",
    "workout_time",
    "average_power",
    "coggan_np",
    "max_power",
    "variability_index",
    "coggan_if",
    "coggan_tss",
]

print("최대 파워가 가장 낮은 기록")

display(
    cleaned_df.nsmallest(
        5,
        "max_power",
    )[max_power_context_columns]
)

print("최대 파워가 가장 높은 기록")

display(
    cleaned_df.nlargest(
        5,
        "max_power",
    )[max_power_context_columns]
)

최대 파워가 가장 낮은 기록


,date,workout_time,average_power,coggan_np,max_power,variability_index,coggan_if,coggan_tss
126,2008-05-18 21:38:31+00:00,311.031,101.00965,101.25792,129.0,1.002458,0.40503,1.41722
203,2009-02-11 18:00:00+00:00,3600.000,195.00000,194.68114,195.0,0.998365,0.70793,50.11669
204,2009-02-12 00:00:00+00:00,3600.000,195.00000,194.68114,195.0,0.998365,0.70793,50.11669
205,2009-02-12 12:00:00+00:00,3600.000,195.00000,194.68114,195.0,0.998365,0.70793,50.11669
202,2009-02-10 18:00:00+00:00,3000.000,200.00000,199.60736,200.0,0.998037,0.72584,43.90424


최대 파워가 가장 높은 기록


,date,workout_time,average_power,coggan_np,max_power,variability_index,coggan_if,coggan_tss
426,2012-07-26 16:51:51+00:00,3591.00,136.84071,207.08743,885.0,1.513347,0.88122,77.46128
130,2008-07-22 16:44:44+00:00,1189.00,250.70227,282.12618,879.0,1.125344,1.12850,42.06163
149,2008-10-01 09:30:23+00:00,4655.08,194.51313,227.83835,862.0,1.171326,0.91135,107.35468
270,2009-05-09 08:48:54+00:00,3755.00,179.60666,242.40953,861.0,1.349669,0.88149,81.04784
429,2012-07-30 17:48:22+00:00,2306.00,140.17736,213.89962,856.0,1.525921,0.91021,53.06902


### 파워 지표 검증 결과

평균 파워가 최대 파워보다 크거나 NP가 최대 파워보다 큰 기록은
없었다. 평균 파워가 NP보다 큰 기록은 4개였지만, 모두 약 50–60분
동안 195W 또는 200W가 일정하게 저장된 연속 기록이었다. NP와의
차이는 0.4W 미만이며, NP 계산에 사용하는 30초 이동평균의 시작
구간 처리로 정확히 설명되므로 이상 기록으로 분류하지 않는다.

평균 파워가 높은 기록은 약 1시간 전후의 고강도 운동으로, NP와
IF 및 TSS를 함께 고려했을 때 가능한 범위였다. 평균 파워가 낮은
기록도 짧은 저강도 이동이나 강한 페달링과 코스팅이 반복된 운동으로
설명할 수 있었다.

`coggan_np / average_power`로 VI를 계산했다. VI 최댓값 2.37은 전체
분포에서 매우 이례적이지만, 해당 기록은 25km에서 약 746m를 오른
라이드로 파워 변화가 큰 산악성 운동일 가능성이 있다. 다른 높은 VI
기록도 낮은 평균 파워와 높은 순간 파워가 함께 나타났다. 따라서 높은
VI는 오류 라벨로 사용하지 않고 라이딩 변동성을 설명하는 파생 지표로
유지한다.

최대 파워 범위는 129–885W였다. 낮은 값은 짧은 저강도 기록과 일정
파워 세션이었고, 높은 값은 IF 약 0.88–1.13의 강한 라이딩이었다.
평균 파워와 NP도 최대 파워 이하였으므로 범위만을 이유로 수정하거나
제외하지 않는다.

## 17. 심박과 케이던스 지표 검증

평균과 최대 지표의 관계를 확인하고, 평균 케이던스의 계산 가중치를
함께 살펴본다.

In [60]:
average_hr_above_max = (
    cleaned_df["average_hr"]
    > cleaned_df["max_heartrate"]
)

print(
    "평균 심박이 최대 심박보다 큰 기록:",
    average_hr_above_max.sum()
)

average_cadence_above_max = (
    cleaned_df["average_cad"]
    > cleaned_df["max_cadence"]
)

print(
    "평균 케이던스가 최대 케이던스보다 큰 기록:",
    average_cadence_above_max.sum()
)

평균 심박이 최대 심박보다 큰 기록: 0
평균 케이던스가 최대 케이던스보다 큰 기록: 0


In [61]:
cleaned_df["average_cad_weight"] = pd.to_numeric(
    analysis_df["average_cad"].map(
        extract_metric_weight
    ),
    errors="coerce",
)

cadence_range_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "average_cad",
    "average_cad_weight",
    "max_cadence",
    "average_power",
    "max_power",
]

print("평균 케이던스가 가장 낮은 기록")

display(
    cleaned_df.nsmallest(
        5,
        "average_cad",
    )[cadence_range_context_columns]
)

print("평균 케이던스가 가장 높은 기록")

display(
    cleaned_df.nlargest(
        5,
        "average_cad",
    )[cadence_range_context_columns]
)

평균 케이던스가 가장 낮은 기록


,date,workout_time,time_riding,average_cad,average_cad_weight,max_cadence,average_power,max_power
53,2007-01-21 09:23:24+00:00,14555.0,10.0,26.00000,2.0,26.0,NaN,NaN
49,2006-12-08 09:51:52+00:00,2395.0,815.0,28.57143,21.0,40.0,NaN,NaN
56,2007-01-23 09:26:08+00:00,5470.0,50.0,35.80000,10.0,41.0,NaN,NaN
228,2009-03-15 11:00:04+00:00,1405.0,1321.0,38.66477,528.0,85.0,37.47568,497.0
353,2010-08-28 12:07:52+00:00,7217.0,6951.0,56.69216,5048.0,168.0,109.89816,713.0


평균 케이던스가 가장 높은 기록


,date,workout_time,time_riding,average_cad,average_cad_weight,max_cadence,average_power,max_power
438,2012-08-25 13:00:13+00:00,3396.0,3393.0,127.00000,1.0,127.0,NaN,NaN
441,2012-09-03 17:35:11+00:00,3095.0,3094.0,127.00000,2.0,127.0,NaN,NaN
32,2006-08-15 12:42:00+00:00,2040.0,1960.0,103.00000,216.0,209.0,NaN,NaN
95,2007-05-11 10:56:02+00:00,4915.0,4680.0,95.97543,936.0,118.0,NaN,NaN
202,2009-02-10 18:00:00+00:00,3000.0,3000.0,95.00000,3000.0,95.0,200.0,200.0


In [62]:
cadence_weight_not_positive = (
    cleaned_df["average_cad_weight"] <= 0
)

print(
    "케이던스 가중치가 0 이하인 기록:",
    cadence_weight_not_positive.sum()
)

cadence_weight_above_riding_time = (
    cleaned_df["average_cad_weight"]
    > cleaned_df["time_riding"]
)

print(
    "케이던스 가중치가 라이딩 시간보다 긴 기록:",
    cadence_weight_above_riding_time.sum()
)

케이던스 가중치가 0 이하인 기록: 0
케이던스 가중치가 라이딩 시간보다 긴 기록: 0


In [63]:
cleaned_df["average_cad_weight_ratio"] = (
    cleaned_df["average_cad_weight"]
    / cleaned_df["time_riding"]
)

cleaned_df["average_cad_weight_ratio"].describe()

count    532.000000
mean       0.794989
std        0.239047
min        0.000295
25%        0.763433
50%        0.897538
75%        0.935966
max        1.000000
Name: average_cad_weight_ratio, dtype: float64

In [64]:
cadence_weight_ratio_needs_review = (
    cleaned_df["average_cad_weight_ratio"]
    < 0.05
)

print(
    "케이던스 반영 비율이 5% 미만인 기록:",
    cadence_weight_ratio_needs_review.sum()
)

cleaned_df["cadence_weight_ratio_needs_review"] = (
    cadence_weight_ratio_needs_review
)

cadence_weight_review_columns = [
    "date",
    "time_riding",
    "average_cad",
    "average_cad_weight",
    "average_cad_weight_ratio",
    "max_cadence",
    "time_relationship_needs_review",
]

cleaned_df.loc[
    cadence_weight_ratio_needs_review,
    cadence_weight_review_columns,
]

케이던스 반영 비율이 5% 미만인 기록: 5


,date,time_riding,average_cad,average_cad_weight,average_cad_weight_ratio,max_cadence,time_relationship_needs_review
6,2005-07-13 18:04:00+00:00,7020.0,75.00000,24.0,0.003419,75.0,False
49,2006-12-08 09:51:52+00:00,815.0,28.57143,21.0,0.025767,40.0,False
110,2007-07-16 05:07:09+00:00,34725.0,66.64572,1448.0,0.041699,106.0,False
438,2012-08-25 13:00:13+00:00,3393.0,127.00000,1.0,0.000295,127.0,False
441,2012-09-03 17:35:11+00:00,3094.0,127.00000,2.0,0.000646,127.0,False


### 심박과 케이던스 지표 검증 결과

평균 심박이 최대 심박보다 크거나 평균 케이던스가 최대 케이던스보다
큰 기록은 없었다. 따라서 평균과 최대 지표의 관계에는 문제가 없다.

평균 케이던스의 극단값을 원본 리스트의 두 번째 값과 함께 확인한
결과, 값 자체보다 계산에 반영된 분량이 중요했다. 낮은 평균 케이던스
상위 기록 중 53번과 56번은 기존 시간 관계 검토 대상이며 케이던스
가중치도 각각 2와 10에 불과했다. 49번의 가중치도 21로 매우 작아
평균 케이던스로 보기 어렵다.

평균 케이던스 127rpm인 438번과 441번은 약 1시간 동안 해당 값을
유지한 기록이라고 볼 수 없으며 계산 가중치도 각각 1과 2뿐이었다.
따라서 극단적인 평균 케이던스는 범위만으로 판단하지 않고
`average_cad_weight`와 함께 해석한다.

케이던스 가중치는 모두 0보다 크고 `time_riding` 이하였으며, 라이딩
시간 대비 가중치 비율의 중앙값은 약 0.90이었다. 다만 가중치가 항상
초를 뜻한다고 단정할 수 없으므로 이 비율을 측정 시간 비율로 직접
해석하지 않는다. 참고용 비율이 5% 미만인
기록은 6번, 49번, 110번, 438번, 441번의 5개였다. 이 기록들은
케이던스 값이 존재하더라도 전체 라이드를 대표하는 평균값으로 보지
않고 `cadence_weight_ratio_needs_review`로 구분한다.

## 18. 심박·파워·NP 계산 가중치 검증

평균 심박, 평균 파워와 NP의 원본 리스트에서 두 번째 값을 계산
가중치로 추출한다. 가중치의 존재 여부와 범위를 확인하고 전체 운동
시간과의 비율을 참고하여 각 평균값이 충분한 근거를 갖는지 살펴본다.
가중치는 지표와 기록 방식에 따라 시간 또는 표본 수처럼 사용될 수
있으므로 측정 시간으로 단정하지 않는다.


In [65]:
metric_columns_with_weight = [
    "average_hr",
    "average_power",
    "coggan_np",
]

for metric_column in metric_columns_with_weight:
    weight_column = f"{metric_column}_weight"

    cleaned_df[weight_column] = pd.to_numeric(
        analysis_df[metric_column].map(
            extract_metric_weight
        ),
        errors="coerce",
    )

In [66]:
cleaned_df[
    [
        "average_hr_weight",
        "average_power_weight",
        "coggan_np_weight",
    ]
].isna().sum()

average_hr_weight       127
average_power_weight    123
coggan_np_weight        123
dtype: int64

In [67]:
hr_metric_without_weight = (
    cleaned_df["average_hr"].notna()
    & cleaned_df["average_hr_weight"].isna()
)

print(
    "평균 심박은 있지만 가중치가 없는 기록:",
    hr_metric_without_weight.sum()
)

power_metric_without_weight = (
    cleaned_df["average_power"].notna()
    & cleaned_df["average_power_weight"].isna()
)

print(
    "평균 파워는 있지만 가중치가 없는 기록:",
    power_metric_without_weight.sum()
)

np_metric_without_weight = (
    cleaned_df["coggan_np"].notna()
    & cleaned_df["coggan_np_weight"].isna()
)

print(
    "NP는 있지만 가중치가 없는 기록:",
    np_metric_without_weight.sum()
)

평균 심박은 있지만 가중치가 없는 기록: 3
평균 파워는 있지만 가중치가 없는 기록: 0
NP는 있지만 가중치가 없는 기록: 0


In [68]:
hr_weight_context_columns = [
    "date",
    "data",
    "average_hr",
    "average_hr_weight",
    "max_heartrate",
    "has_hr_sensor",
    "hr_metric_without_sensor",
]

cleaned_df.loc[
    hr_metric_without_weight,
    hr_weight_context_columns,
]

,date,data,average_hr,average_hr_weight,max_heartrate,has_hr_sensor,hr_metric_without_sensor
392,2012-01-04 15:35:51+00:00,---------------,148.0,NaN,NaN,False,True
393,2012-01-05 12:32:51+00:00,---------------,145.0,NaN,NaN,False,True
394,2012-01-06 13:22:53+00:00,---------------,145.0,NaN,NaN,False,True


In [69]:
weight_columns = [
    "average_hr_weight",
    "average_power_weight",
    "coggan_np_weight",
]

print("가중치가 0 이하인 기록")

display(
    (cleaned_df[weight_columns] <= 0).sum()
)

print("가중치가 전체 운동 시간보다 긴 기록")

display(
    cleaned_df[weight_columns].gt(
        cleaned_df["workout_time"],
        axis="index",
    ).sum()
)

가중치가 0 이하인 기록


average_hr_weight       0
average_power_weight    0
coggan_np_weight        0
dtype: int64

가중치가 전체 운동 시간보다 긴 기록


average_hr_weight       0
average_power_weight    0
coggan_np_weight        2
dtype: int64

In [70]:
np_weight_above_workout_time = (
    cleaned_df["coggan_np_weight"]
    > cleaned_df["workout_time"]
)

np_weight_review_columns = [
    "date",
    "workout_time",
    "time_riding",
    "coggan_np",
    "coggan_np_weight",
    "coggan_if",
    "coggan_tss",
    "time_relationship_needs_review",
]

cleaned_df.loc[
    np_weight_above_workout_time,
    np_weight_review_columns,
]

,date,workout_time,time_riding,coggan_np,coggan_np_weight,coggan_if,coggan_tss,time_relationship_needs_review
100,2007-06-02 23:00:00+00:00,14908.060,14651.280,244.37967,14908.320,0.88865,327.03255,False
115,2007-11-19 18:43:45+00:00,3605.602,3657.408,147.81254,3682.176,0.53750,29.55012,True


In [71]:
cleaned_df["average_hr_weight_ratio"] = (
    cleaned_df["average_hr_weight"]
    / cleaned_df["workout_time"]
)

cleaned_df["average_power_weight_ratio"] = (
    cleaned_df["average_power_weight"]
    / cleaned_df["workout_time"]
)

cleaned_df["coggan_np_weight_ratio"] = (
    cleaned_df["coggan_np_weight"]
    / cleaned_df["workout_time"]
)

weight_ratio_columns = [
    "average_hr_weight_ratio",
    "average_power_weight_ratio",
    "coggan_np_weight_ratio",
]

cleaned_df[weight_ratio_columns].describe()

,average_hr_weight_ratio,average_power_weight_ratio,coggan_np_weight_ratio
count,465.000000,469.000000,469.000000
mean,0.816488,0.961564,0.993722
std,0.311577,0.091174,0.022152
min,0.000454,0.200000,0.793745
25%,0.790616,0.989571,0.998029
50%,0.997790,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.021238


In [72]:
hr_weight_ratio_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "average_hr",
    "average_hr_weight",
    "average_hr_weight_ratio",
    "max_heartrate",
    "has_hr_sensor",
    "time_relationship_needs_review",
]

print("심박 가중치의 비율이 가장 낮은 기록")

display(
    cleaned_df.nsmallest(
        5,
        "average_hr_weight_ratio",
    )[hr_weight_ratio_context_columns]
)

print("파워 가중치의 비율이 가장 낮은 기록")

power_weight_ratio_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "average_power",
    "average_power_weight",
    "average_power_weight_ratio",
    "coggan_np",
    "max_power",
    "time_relationship_needs_review",
]

display(
    cleaned_df.nsmallest(
        5,
        "average_power_weight_ratio",
    )[power_weight_ratio_context_columns]
)

심박 가중치의 비율이 가장 낮은 기록


,date,workout_time,time_riding,average_hr,average_hr_weight,average_hr_weight_ratio,max_heartrate,has_hr_sensor,time_relationship_needs_review
577,2016-07-31 09:43:57+00:00,6605.0,6422.0,162.0,3.0,0.000454,162.0,True,False
576,2016-07-23 23:44:16+00:00,5733.0,5733.0,123.0,3.0,0.000523,123.0,True,False
14,2005-09-11 22:40:00+00:00,6120.0,4620.0,14.0,4.0,0.000654,14.0,True,False
12,2005-08-28 23:06:00+00:00,1800.0,1780.0,14.0,4.0,0.002222,14.0,True,False
42,2006-10-28 14:13:00+00:00,6360.0,5980.0,26.0,72.0,0.011321,70.0,True,False


파워 가중치의 비율이 가장 낮은 기록


,date,workout_time,time_riding,average_power,average_power_weight,average_power_weight_ratio,coggan_np,max_power,time_relationship_needs_review
96,2007-05-13 05:20:15+00:00,30275.00,30090.00,190.12172,6055.0,0.200000,246.28281,590.000,False
107,2007-07-03 12:18:18+00:00,3720.00,3715.00,248.23628,744.0,0.200000,257.86354,474.028,False
117,2007-11-22 13:54:41+00:00,6289.12,5236.56,182.36804,4312.0,0.685629,212.77059,582.000,False
120,2007-11-25 09:04:29+00:00,18540.86,14660.10,141.37886,12944.0,0.698134,192.54259,724.000,False
140,2008-09-10 12:20:33+00:00,13477.66,11704.14,159.09193,9572.0,0.710212,214.19594,603.000,False


In [73]:
hr_weight_needs_review = (
    cleaned_df["average_hr_weight"] < 100
)

print(
    "심박 가중치가 100 미만인 기록:",
    hr_weight_needs_review.sum()
)

cleaned_df["hr_weight_needs_review"] = (
    hr_weight_needs_review
)

심박 가중치가 100 미만인 기록: 5


### 심박·파워·NP 계산 가중치 검증 결과

평균 파워와 NP는 값이 존재하는 469개 기록 모두 가중치도 존재했다.
평균 심박은 값이 존재하지만 가중치가 없는 기록이 3개였으며, 앞서
확인한 심박 센서 없이 평균 심박이 직접 입력된 기록과 일치했다.
따라서 가중치 누락에 대한 별도 품질 라벨은 추가하지 않는다.

세 가중치는 모두 0보다 컸다. 심박과 평균 파워 가중치는 전체 운동
시간 이하였고, NP 가중치가 전체 운동 시간보다 큰 기록은 2개였다.
100번의 차이는 수치상 0.26으로 계산 정밀도 수준이며, 115번은 이미
`time_relationship_needs_review`로 구분된 시간 관계 이상 기록이었다.

가중치와 전체 운동 시간의 비율은 측정 범위를 직접 의미하지 않는다.
96번과 107번의 평균 파워 가중치 비율 0.2는 운동 시간과 가중치가
정확히 5:1이므로 5초 간격 기록으로도 설명할 수 있다. 따라서 낮은
비율만으로 기록을 제외하지 않는다. 반면 평균 심박 가중치가 3, 4,
72처럼 매우 작은 5개 기록은 값의 대표성이 낮을 수 있으므로 현재
데이터에서 100 미만을 `hr_weight_needs_review`로 구분한다. 이 기준은
자동 삭제 조건이 아니라 추가 검토를 위한 프로젝트 기준이다.


## 19. 획득 고도 검증

획득 고도의 결측값과 범위를 확인하고, 거리당 획득 고도를 계산하여
극단적인 값이 고도 자체의 문제인지 기존 거리·시간 문제에서 파생된
결과인지 살펴본다.


In [74]:
elevation_gain_missing = (
    cleaned_df["elevation_gain"].isna()
)

print(
    "획득 고도가 결측값인 기록:",
    elevation_gain_missing.sum()
)

elevation_gain_zero = (
    cleaned_df["elevation_gain"] == 0
)

print(
    "획득 고도가 0인 기록:",
    elevation_gain_zero.sum()
)

elevation_gain_negative = (
    cleaned_df["elevation_gain"] < 0
)

print(
    "획득 고도가 음수인 기록:",
    elevation_gain_negative.sum()
)

cleaned_df["elevation_gain"].describe()

획득 고도가 결측값인 기록: 96
획득 고도가 0인 기록: 0
획득 고도가 음수인 기록: 0


count     496.000000
mean      445.868522
std       472.184718
min         3.004000
25%       247.960250
50%       300.958000
75%       470.925000
max      4888.000000
Name: elevation_gain, dtype: float64

In [75]:
elevation_gain_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "total_distance",
    "elevation_gain",
    "average_speed",
    "average_power",
    "coggan_tss",
]

print("획득 고도가 가장 낮은 기록")

display(
    cleaned_df.nsmallest(
        5,
        "elevation_gain",
    )[elevation_gain_context_columns]
)

print("획득 고도가 가장 높은 기록")

display(
    cleaned_df.nlargest(
        5,
        "elevation_gain",
    )[elevation_gain_context_columns]
)

획득 고도가 가장 낮은 기록


,date,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,coggan_tss
562,2015-07-21 11:19:33+00:00,126.0,123.0,0.60574,3.004,17.72912,126.90476,1.53483
45,2006-11-12 20:31:06+00:00,3860.0,3550.0,NaN,4.000,NaN,NaN,NaN
48,2006-12-02 11:08:05+00:00,3780.0,3500.0,NaN,4.000,NaN,NaN,NaN
74,2007-03-18 21:02:30+00:00,1505.0,1495.0,0.00700,4.000,5.04000,NaN,NaN
61,2007-02-13 06:47:52+00:00,3460.0,NaN,NaN,12.000,NaN,NaN,NaN


획득 고도가 가장 높은 기록


,date,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,coggan_tss
313,2009-07-04 05:09:48+00:00,40169.0,35779.0,175.69200,4888.0,17.74418,132.64348,459.14827
110,2007-07-16 05:07:09+00:00,36570.0,34725.0,197.17967,4415.0,20.44195,NaN,NaN
96,2007-05-13 05:20:15+00:00,30275.0,30090.0,175.16700,3232.0,22.06056,190.12172,674.50390
78,2007-04-08 08:31:03+00:00,24560.0,23955.0,166.78500,2465.4,25.06475,NaN,NaN
98,2007-05-20 05:59:16+00:00,28815.0,27350.0,182.16000,2337.0,23.97718,NaN,NaN


In [76]:
cleaned_df["elevation_gain_per_km"] = (
    cleaned_df["elevation_gain"]
    / cleaned_df["total_distance"]
)

cleaned_df["elevation_gain_per_km"].describe()

count       481.000000
mean        694.887727
std       14977.790370
min           1.911896
25%           7.136895
50%           8.095228
75%          11.512835
max      328500.000000
Name: elevation_gain_per_km, dtype: float64

In [77]:
elevation_gain_per_km_context_columns = [
    "date",
    "workout_time",
    "time_riding",
    "total_distance",
    "elevation_gain",
    "elevation_gain_per_km",
    "average_speed",
    "time_relationship_needs_review",
    "speed_relationship_needs_review",
]

cleaned_df.nlargest(
    10,
    "elevation_gain_per_km",
)[elevation_gain_per_km_context_columns]

,date,workout_time,time_riding,total_distance,elevation_gain,elevation_gain_per_km,average_speed,time_relationship_needs_review,speed_relationship_needs_review
56,2007-01-23 09:26:08+00:00,5470.0,50.0,0.00200,657.0,328500.000000,1.44000,True,True
74,2007-03-18 21:02:30+00:00,1505.0,1495.0,0.00700,4.0,571.428571,5.04000,False,True
49,2006-12-08 09:51:52+00:00,2395.0,815.0,0.93786,134.0,142.878468,4.19416,False,False
107,2007-07-03 12:18:18+00:00,3720.0,3715.0,12.02500,1026.0,85.322245,11.65276,False,False
108,2007-07-05 13:59:16+00:00,6425.0,6420.0,21.00844,1593.0,75.826668,11.78044,False,False
17,2005-11-21 00:48:00+00:00,7200.0,1060.0,5.08000,379.0,74.606299,17.25283,False,False
65,2007-02-23 06:56:21+00:00,6395.0,5515.0,7.31222,444.0,60.720274,13.96499,False,True
30,2006-07-15 17:54:00+00:00,4680.0,4500.0,8.39000,360.0,42.908224,11.61692,False,True
207,2009-02-14 13:53:09+00:00,1223.0,1087.0,5.05000,173.6,34.376238,16.74033,False,False
29,2006-07-14 17:42:00+00:00,5160.0,4560.0,7.98950,269.0,33.669191,10.81286,False,True


### 획득 고도 검증 결과

획득 고도는 96개 기록에서 결측이었고, 값이 존재하는 496개 기록은
모두 양수였다. 범위는 약 3–4,888m였다. 낮은 값은 거리 결측 또는
매우 짧은 기록에서 나타났고, 높은 값은 약 166–197km를 7시간 이상
주행한 장거리 라이딩이므로 산악 코스로 설명할 수 있었다.

거리당 획득 고도 `elevation_gain_per_km`의 중앙값은 약 8.1m/km,
75% 지점은 약 11.5m/km였다. 최댓값 328,500m/km인 56번은 거리가
0.002km로 저장된 기존 시간·속도 검토 대상이며, 74번도 0.007km의
거리 문제로 비율이 커진 기존 속도 검토 대상이었다. 49번은 약
0.94km에서 134m를 오른 짧고 가파른 부분 기록으로 설명할 수 있다.

따라서 극단적인 비율은 획득 고도만의 새로운 오류로 보지 않고 기존
시간·속도 품질 라벨과 함께 해석한다. 고도 전용 검토 라벨은 추가하지
않으며 `elevation_gain_per_km`는 코스 특성을 나타내는 파생 지표로
유지한다.


## 20. 품질 검토 라벨 종합

주요 지표 검증에서 만든 검토 라벨을 한곳에 모으고 각 라벨에 해당하는
기록 수를 확인한다. 검토 라벨은 오류 확정이나 자동 삭제를 의미하지
않으며, 이후 분석 목적에 따라 해당 지표의 사용 여부를 판단하기 위한
근거로 사용한다.


In [78]:
review_flag_columns = [
    "hr_metric_without_sensor",
    "tss_without_power_sensor",
    "hr_range_needs_review",
    "reference_power_differs_from_cp",
    "time_relationship_needs_review",
    "speed_relationship_needs_review",
    "cadence_weight_ratio_needs_review",
    "hr_weight_needs_review",
]

cleaned_df[
    review_flag_columns
].sum().sort_values(
    ascending=False
)

reference_power_differs_from_cp      90
speed_relationship_needs_review      22
tss_without_power_sensor             21
hr_range_needs_review                 5
cadence_weight_ratio_needs_review     5
hr_weight_needs_review                5
hr_metric_without_sensor              3
time_relationship_needs_review        3
dtype: int64

### 품질 검토 라벨 개수

기준 파워와 저장된 IF에서 역산한 기준 파워가 다른 기록이 90개로
가장 많았다. 속도 관계 검토 대상은 22개, 파워 센서 표시 없이 TSS가
저장된 기록은 21개였다. 심박 범위, 케이던스 가중치 비율과 심박
가중치 검토 대상은 각각 5개였고, 심박 센서 표시 없이 평균 심박이
저장된 기록과 시간 관계 검토 대상은 각각 3개였다.

이 개수들은 서로 다른 기록 수를 뜻하지 않는다. 한 라이드가 여러 검토
라벨을 동시에 가질 수 있으므로 단순히 합산하지 않는다. 다음 단계에서
라벨 간 중복을 확인한 뒤 분석 목적별 데이터 사용 기준을 정한다.


In [79]:
cleaned_df["review_flag_count"] = (
    cleaned_df[review_flag_columns].sum(
        axis="columns"
    )
)

cleaned_df["review_flag_count"].value_counts().sort_index()

review_flag_count
0    451
1    128
2     13
Name: count, dtype: int64

In [80]:
multiple_review_flags = (
    cleaned_df["review_flag_count"] >= 2
)

print(
    "검토 라벨이 2개 이상인 기록:",
    multiple_review_flags.sum()
)

multiple_review_flag_columns = (
    ["date", "review_flag_count"]
    + review_flag_columns
)

cleaned_df.loc[
    multiple_review_flags,
    multiple_review_flag_columns,
]

검토 라벨이 2개 이상인 기록: 13


,date,review_flag_count,hr_metric_without_sensor,tss_without_power_sensor,hr_range_needs_review,reference_power_differs_from_cp,time_relationship_needs_review,speed_relationship_needs_review,cadence_weight_ratio_needs_review,hr_weight_needs_review
12,2005-08-28 23:06:00+00:00,2,False,False,True,False,False,False,False,True
14,2005-09-11 22:40:00+00:00,2,False,False,True,False,False,False,False,True
42,2006-10-28 14:13:00+00:00,2,False,False,True,False,False,False,False,True
56,2007-01-23 09:26:08+00:00,2,False,False,False,False,True,True,False,False
96,2007-05-13 05:20:15+00:00,2,False,False,False,True,False,True,False,False
385,2010-10-23 16:36:08+00:00,2,False,False,False,True,False,True,False,False
392,2012-01-04 15:35:51+00:00,2,True,True,False,False,False,False,False,False
393,2012-01-05 12:32:51+00:00,2,True,True,False,False,False,False,False,False
394,2012-01-06 13:22:53+00:00,2,True,True,False,False,False,False,False,False
438,2012-08-25 13:00:13+00:00,2,False,True,False,False,False,False,True,False


### 품질 검토 라벨 중복 결과

전체 592개 라이드 중 검토 라벨이 없는 기록은 451개, 1개인 기록은
128개, 2개인 기록은 13개였다. 3개 이상 겹친 기록은 없었다. 따라서
대부분의 라이드는 주요 지표 관계에서 검토 사항이 없거나 한 가지뿐이며,
여러 문제가 복잡하게 겹친 기록은 많지 않았다.

2개 라벨이 겹친 조합은 심박 범위와 심박 가중치 3개, 시간과 속도
관계 1개, 기준 파워 불일치와 속도 관계 2개, 심박 센서 없이 심박이
존재하면서 파워 센서 없이 TSS가 존재한 기록 3개, 파워 센서 없이
TSS가 존재하면서 케이던스 가중치 비율이 낮은 기록 2개, 기준 파워
불일치와 심박 가중치가 겹친 기록 2개였다.

`review_flag_count`는 라이드별 검토 사항의 수를 나타내는 요약 지표로
유지한다. 다음 단계에서는 라이드 전체를 일괄 제외하지 않고, 각 라벨이
영향을 주는 지표만 구분하여 분석 목적별 사용 기준을 정한다.


## 21. 분석 목적별 데이터 사용 기준

품질 검토 라벨이 있는 라이드를 일괄 삭제하지 않고, 각 라벨이 영향을
주는 지표만 구분하여 분석에 사용할 수 있는지 판단한다. 먼저 심박
분석에 필요한 평균·최대 심박의 존재 여부와 관련 품질 라벨을 결합한다.


### 심박 분석 사용 기준

심박 센서 표시 없이 평균 심박이 존재하거나, 심박 범위가 비정상적이거나,
평균 심박 계산 가중치가 지나치게 작은 기록을 심박 분석 검토 대상으로
구분한다. 이 조건 중 하나 이상에 해당하는 기록은 10개였다.


In [81]:
cleaned_df["hr_analysis_needs_review"] = (
    cleaned_df["hr_metric_without_sensor"]
    | cleaned_df["hr_range_needs_review"]
    | cleaned_df["hr_weight_needs_review"]
)

print(
    "심박 분석 검토 대상:",
    cleaned_df["hr_analysis_needs_review"].sum()
)

심박 분석 검토 대상: 10


In [82]:
hr_analysis_review_columns = [
    "date",
    "data",
    "average_hr",
    "average_hr_weight",
    "max_heartrate",
    "has_hr_sensor",
    "hr_metric_without_sensor",
    "hr_range_needs_review",
    "hr_weight_needs_review",
    "hr_analysis_needs_review",
]

cleaned_df.loc[
    cleaned_df["hr_analysis_needs_review"],
    hr_analysis_review_columns,
]

,date,data,average_hr,average_hr_weight,max_heartrate,has_hr_sensor,hr_metric_without_sensor,hr_range_needs_review,hr_weight_needs_review,hr_analysis_needs_review
12,2005-08-28 23:06:00+00:00,TDS-H--A-L-----,14.00000,4.0,14.0,True,False,True,True,True
14,2005-09-11 22:40:00+00:00,TDS-H--A-L-E---,14.00000,4.0,14.0,True,False,True,True,True
42,2006-10-28 14:13:00+00:00,TDS-H--A-L-----,26.00000,72.0,70.0,True,False,True,True,True
102,2007-06-04 17:56:03+00:00,T---H--A-------,136.38401,1013.0,228.0,True,False,True,False,True
317,2009-07-16 16:01:07+00:00,TDSPHC-AGL-----,141.29711,6220.0,229.0,True,False,True,False,True
392,2012-01-04 15:35:51+00:00,---------------,148.00000,NaN,NaN,False,True,False,False,True
393,2012-01-05 12:32:51+00:00,---------------,145.00000,NaN,NaN,False,True,False,False,True
394,2012-01-06 13:22:53+00:00,---------------,145.00000,NaN,NaN,False,True,False,False,True
576,2016-07-23 23:44:16+00:00,TDSPHC-A-L-E---,123.00000,3.0,123.0,True,False,False,True,True
577,2016-07-31 09:43:57+00:00,TDSPHC-AGL-E---,162.00000,3.0,162.0,True,False,False,True,True


In [83]:
cleaned_df["hr_analysis_is_usable"] = (
    ~cleaned_df["hr_analysis_needs_review"]
    & cleaned_df["average_hr"].notna()
    & cleaned_df["max_heartrate"].notna()
)

print(
    "심박 분석에 사용할 수 있는 기록:",
    cleaned_df["hr_analysis_is_usable"].sum()
)

심박 분석에 사용할 수 있는 기록: 458


### 심박 분석 사용 기준 결과

평균 심박과 최대 심박이 모두 존재하고 심박 품질 검토 대상이 아닌
기록을 `hr_analysis_is_usable`로 표시했다. 전체 592개 라이드 중
458개를 심박 분석에 사용할 수 있으며, 나머지 134개는 심박 지표가
결측이거나 품질 검토가 필요한 기록이다.

사용할 수 없는 심박 지표가 있다고 해서 해당 라이드 전체를 제외하지
않는다. 시간·거리·파워 등 다른 지표는 각각의 사용 기준에 따라 별도로
판단한다.


### 시간과 평균 속도 분석 사용 기준

전체 운동 시간, 실제 라이딩 시간과 평균 속도는 서로 다른 용도로
사용하므로 각각 사용 가능 여부를 구분한다. 시간값은 존재하며 양수여야
하고, 라이딩 시간은 기존 시간 관계 검토 대상이 아니어야 한다. 평균
속도는 사용할 수 있는 라이딩 시간과 양수인 거리·속도가 존재하며 기존
속도 관계 검토 대상이 아닌 기록만 사용한다.


In [84]:
cleaned_df["workout_time_is_usable"] = (
    cleaned_df["workout_time"].notna()
    & (cleaned_df["workout_time"] > 0)
)

cleaned_df["riding_time_is_usable"] = (
    cleaned_df["time_riding"].notna()
    & (cleaned_df["time_riding"] > 0)
    & ~cleaned_df["time_relationship_needs_review"]
)

print(
    "전체 운동 시간 분석에 사용할 수 있는 기록:",
    cleaned_df["workout_time_is_usable"].sum()
)

print(
    "라이딩 시간 분석에 사용할 수 있는 기록:",
    cleaned_df["riding_time_is_usable"].sum()
)

전체 운동 시간 분석에 사용할 수 있는 기록: 591
라이딩 시간 분석에 사용할 수 있는 기록: 571


In [85]:
cleaned_df["speed_analysis_is_usable"] = (
    cleaned_df["riding_time_is_usable"]
    & cleaned_df["total_distance"].notna()
    & (cleaned_df["total_distance"] > 0)
    & cleaned_df["average_speed"].notna()
    & (cleaned_df["average_speed"] > 0)
    & ~cleaned_df["speed_relationship_needs_review"]
)

print(
    "평균 속도 분석에 사용할 수 있는 기록:",
    cleaned_df["speed_analysis_is_usable"].sum()
)

평균 속도 분석에 사용할 수 있는 기록: 540


### 시간과 평균 속도 분석 사용 기준 결과

전체 운동 시간은 591개 기록에서 사용할 수 있었다. 라이딩 시간은
결측값 18개와 시간 관계 검토 대상 3개를 제외한 571개 기록에서
사용할 수 있었다. 전체 운동 시간과 라이딩 시간의 사용 가능 기록 수가
20개 차이 나는 이유는 두 시간이 모두 결측인 403번이 양쪽에 함께
포함되고, 나머지 라이딩 시간 결측 및 관계 이상 기록만 추가로 제외되기
때문이다.

평균 속도는 사용할 수 있는 라이딩 시간, 양수인 거리와 속도, 거리·속도
관계 기준을 모두 만족한 540개 기록에서 사용할 수 있었다. 각 사용 가능
열은 해당 지표의 분석에만 적용하며 다른 지표까지 일괄 제외하지 않는다.


### 파워·강도·훈련 부하 분석 사용 기준

기본 파워 지표, 파워 센서 기반 IF·TSS, 출처와 관계없이 저장된 TSS,
현재 `cp_setting`을 기준으로 비교할 수 있는 강도를 서로 다른 용도로
구분한다. 기준 파워 불일치는 평균 파워·최대 파워·NP나 저장된 IF·TSS
자체를 제외하는 조건으로 사용하지 않고, `cp_setting`과 직접 비교하는
분석에서만 제외한다.


In [86]:
cleaned_df["power_analysis_is_usable"] = (
    cleaned_df["has_power_sensor"]
    & cleaned_df["average_power"].notna()
    & (cleaned_df["average_power"] > 0)
    & cleaned_df["max_power"].notna()
    & (cleaned_df["max_power"] > 0)
    & cleaned_df["coggan_np"].notna()
    & (cleaned_df["coggan_np"] > 0)
)

print(
    "파워 분석에 사용할 수 있는 기록:",
    cleaned_df["power_analysis_is_usable"].sum()
)

파워 분석에 사용할 수 있는 기록: 469


In [87]:
cleaned_df["power_load_analysis_is_usable"] = (
    cleaned_df["power_analysis_is_usable"]
    & cleaned_df["coggan_if"].notna()
    & (cleaned_df["coggan_if"] > 0)
    & cleaned_df["coggan_tss"].notna()
    & (cleaned_df["coggan_tss"] > 0)
)

print(
    "파워 기반 부하 분석에 사용할 수 있는 기록:",
    cleaned_df["power_load_analysis_is_usable"].sum()
)

파워 기반 부하 분석에 사용할 수 있는 기록: 469


In [88]:
cleaned_df["stored_tss_is_usable"] = (
    cleaned_df["coggan_tss"].notna()
    & (cleaned_df["coggan_tss"] > 0)
)

print(
    "저장된 TSS 분석에 사용할 수 있는 기록:",
    cleaned_df["stored_tss_is_usable"].sum()
)

저장된 TSS 분석에 사용할 수 있는 기록: 490


In [89]:
cleaned_df["cp_based_intensity_analysis_is_usable"] = (
    cleaned_df["power_load_analysis_is_usable"]
    & cleaned_df["cp_setting"].notna()
    & (cleaned_df["cp_setting"] > 0)
    & ~cleaned_df["reference_power_differs_from_cp"]
)

print(
    "CP 기준 강도 분석에 사용할 수 있는 기록:",
    cleaned_df[
        "cp_based_intensity_analysis_is_usable"
    ].sum()
)

CP 기준 강도 분석에 사용할 수 있는 기록: 379


### 파워·강도·훈련 부하 분석 사용 기준 결과

파워 센서가 있고 평균 파워·최대 파워·NP가 존재하며 양수인 469개
기록을 `power_analysis_is_usable`로 표시했다. 이 469개는 IF와 TSS도
존재하고 양수이므로 파워 센서 기반 부하 분석에도 사용할 수 있다.

저장된 TSS가 존재하고 양수인 기록은 490개였다. 이 중 파워 센서가
없는 21개는 파워 기반 TSS라고 확정하지 않지만, 전체 훈련 부하 흐름을
볼 때 참고할 수 있도록 `stored_tss_is_usable`에 포함한다. 센서 기반
분석에는 `power_load_analysis_is_usable` 469개만 사용한다.

파워 기반 부하 기록 중 현재 `cp_setting`과 저장된 IF의 기준 파워가
일치하는 379개만 `cp_based_intensity_analysis_is_usable`로 표시했다.
기준 파워가 다른 90개는 저장된 IF·TSS 흐름에서는 유지하지만, NP와
현재 `cp_setting`을 직접 이용한 강도 비교에서는 제외한다.


### 거리와 획득 고도 분석 사용 기준

총거리는 값이 존재하고 양수이며 기존 거리·속도 관계 검토 대상이 아닌
기록만 사용한다. 획득 고도는 값이 존재하고 양수인 기록을 사용하며,
거리당 획득 고도는 거리와 획득 고도의 개별 사용 기준을 모두 만족한
기록만 사용한다.


In [90]:
cleaned_df["distance_analysis_is_usable"] = (
    cleaned_df["total_distance"].notna()
    & (cleaned_df["total_distance"] > 0)
    & ~cleaned_df["speed_relationship_needs_review"]
)

cleaned_df["elevation_analysis_is_usable"] = (
    cleaned_df["elevation_gain"].notna()
    & (cleaned_df["elevation_gain"] > 0)
)

cleaned_df["elevation_per_distance_analysis_is_usable"] = (
    cleaned_df["elevation_analysis_is_usable"]
    & cleaned_df["distance_analysis_is_usable"]
    & cleaned_df["elevation_gain_per_km"].notna()
    & (cleaned_df["elevation_gain_per_km"] > 0)
)

print(
    "거리 분석에 사용할 수 있는 기록:",
    cleaned_df["distance_analysis_is_usable"].sum()
)

print(
    "획득 고도 분석에 사용할 수 있는 기록:",
    cleaned_df["elevation_analysis_is_usable"].sum()
)

print(
    "거리당 획득 고도 분석에 사용할 수 있는 기록:",
    cleaned_df[
        "elevation_per_distance_analysis_is_usable"
    ].sum()
)

거리 분석에 사용할 수 있는 기록: 541
획득 고도 분석에 사용할 수 있는 기록: 496
거리당 획득 고도 분석에 사용할 수 있는 기록: 471


### 거리와 획득 고도 분석 사용 기준 결과

총거리는 결측값과 거리·속도 관계 검토 대상을 제외한 541개 기록에서
사용할 수 있었다. 획득 고도는 값이 존재하고 양수인 496개 기록에서
사용할 수 있었으며, 두 조건을 모두 만족하는 거리당 획득 고도는
471개 기록에서 사용할 수 있었다.

거리와 획득 고도는 라이드 전체의 운동량과 코스 특성을 나타내지만,
거리당 획득 고도처럼 두 지표를 결합할 때는 양쪽의 사용 기준을 모두
만족해야 한다.


### 케이던스 분석 사용 기준

평균·최대 케이던스가 모두 존재하고 양수이며, 평균 케이던스 계산
가중치 비율이 지나치게 낮은 검토 대상이 아닌 기록만 사용한다.


In [91]:
cleaned_df["cadence_analysis_is_usable"] = (
    cleaned_df["average_cad"].notna()
    & (cleaned_df["average_cad"] > 0)
    & cleaned_df["max_cadence"].notna()
    & (cleaned_df["max_cadence"] > 0)
    & ~cleaned_df["cadence_weight_ratio_needs_review"]
)

print(
    "케이던스 분석에 사용할 수 있는 기록:",
    cleaned_df["cadence_analysis_is_usable"].sum()
)

케이던스 분석에 사용할 수 있는 기록: 527


### 케이던스 분석 사용 기준 결과

평균·최대 케이던스가 존재하는 532개 기록 중 계산 가중치 비율 검토
대상 5개를 제외한 527개를 `cadence_analysis_is_usable`로 표시했다.
케이던스를 사용할 수 없어도 해당 라이드의 시간·거리·파워 등 다른
지표는 각각의 사용 기준에 따라 유지한다.

이로써 현재 선택한 주요 지표의 분석 목적별 사용 기준을 모두 정의했다.


## 22. 최종 분석 열 정리

정제 과정에서 만든 열을 기본 정보와 주요 지표, 분석용 파생 지표, 계산
가중치, 품질 라벨, 분석 사용 가능 여부의 다섯 그룹으로 정리한다. 검증
과정에서만 사용한 임시 계산 열은 제외하고 이후 시각화와 분석에 필요한
열만 선택하여 `final_df`를 만든다.


In [92]:
ride_info_columns = [
    "date",
    "sport",
    "data",
]

base_analysis_columns = (
    ride_info_columns
    + selected_metric_columns
)

print(
    "기본 분석 열 개수:",
    len(base_analysis_columns)
)

cleaned_df[
    base_analysis_columns
].head()

기본 분석 열 개수: 18


,date,sport,data,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,2005-06-25 14:26:00+00:00,Bike,TDS-H--A-L-----,4800.0,4780.0,35.3275,367.0,26.60649,NaN,NaN,NaN,200.0,146.91667,184.0,NaN,NaN,NaN,NaN
1,2005-06-27 08:39:00+00:00,Bike,TDS-H--A-L-----,6476.0,6325.0,35.0140,467.5,19.92892,NaN,NaN,NaN,200.0,124.97267,182.0,NaN,NaN,NaN,NaN
2,2005-06-28 17:44:00+00:00,Bike,TDS-HC-A-L-----,2156.0,2156.0,17.6530,92.0,29.71052,NaN,NaN,NaN,200.0,157.04592,168.0,88.72929,98.0,NaN,NaN
3,2005-07-06 18:02:00+00:00,Bike,TDS-H--A-L-----,6360.0,6320.0,31.0495,512.0,17.68642,NaN,NaN,NaN,200.0,120.05660,182.0,NaN,NaN,NaN,NaN
4,2005-07-10 09:39:00+00:00,Bike,TDS-H--A-L-----,4680.0,4660.0,32.7700,471.0,25.31588,NaN,NaN,NaN,200.0,146.42735,177.0,NaN,NaN,NaN,NaN


In [93]:
derived_metric_columns = [
    "workout_hours",
    "stopped_time_minutes",
    "riding_time_ratio",
    "variability_index",
    "elevation_gain_per_km",
    "reference_power_from_stored_if",
]

print(
    "파생 지표 열 개수:",
    len(derived_metric_columns)
)

cleaned_df[
    derived_metric_columns
].head()

파생 지표 열 개수: 6


,workout_hours,stopped_time_minutes,riding_time_ratio,variability_index,elevation_gain_per_km,reference_power_from_stored_if
0,1.333333,0.333333,0.995833,NaN,10.388508,NaN
1,1.798889,2.516667,0.976683,NaN,13.351802,NaN
2,0.598889,0.000000,1.000000,NaN,5.211579,NaN
3,1.766667,0.666667,0.993711,NaN,16.489799,NaN
4,1.300000,0.333333,0.995726,NaN,14.372902,NaN


In [94]:
calculation_weight_columns = [
    "average_power_weight",
    "coggan_np_weight",
    "average_hr_weight",
    "average_cad_weight",
    "coggan_if_weight",
    "average_power_weight_ratio",
    "coggan_np_weight_ratio",
    "average_hr_weight_ratio",
    "average_cad_weight_ratio",
]

print(
    "계산 가중치 열 개수:",
    len(calculation_weight_columns)
)

cleaned_df[
    calculation_weight_columns
].head()

계산 가중치 열 개수: 9


,average_power_weight,coggan_np_weight,average_hr_weight,average_cad_weight,coggan_if_weight,average_power_weight_ratio,coggan_np_weight_ratio,average_hr_weight_ratio,average_cad_weight_ratio
0,NaN,NaN,960.0,NaN,NaN,NaN,NaN,0.2,NaN
1,NaN,NaN,6476.0,NaN,NaN,NaN,NaN,1.0,NaN
2,NaN,NaN,2156.0,1919.0,NaN,NaN,NaN,1.0,0.890074
3,NaN,NaN,1272.0,NaN,NaN,NaN,NaN,0.2,NaN
4,NaN,NaN,936.0,NaN,NaN,NaN,NaN,0.2,NaN


In [95]:
quality_flag_columns = (
    [
        "has_power_sensor",
        "has_hr_sensor",
        "has_power_metric",
        "has_hr_metric",
        "has_tss_metric",
        "workout_time_missing",
        "riding_time_missing",
        "distance_speed_missing",
    ]
    + review_flag_columns
    + [
        "review_flag_count",
        "hr_analysis_needs_review",
    ]
)

print(
    "품질 관련 열 개수:",
    len(quality_flag_columns)
)

cleaned_df[
    quality_flag_columns
].head()

품질 관련 열 개수: 18


,has_power_sensor,has_hr_sensor,has_power_metric,has_hr_metric,has_tss_metric,workout_time_missing,riding_time_missing,distance_speed_missing,hr_metric_without_sensor,tss_without_power_sensor,hr_range_needs_review,reference_power_differs_from_cp,time_relationship_needs_review,speed_relationship_needs_review,cadence_weight_ratio_needs_review,hr_weight_needs_review,review_flag_count,hr_analysis_needs_review
0,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,0,False
1,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,0,False
2,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,0,False
3,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,0,False
4,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,0,False


In [96]:
analysis_usability_columns = [
    "workout_time_is_usable",
    "riding_time_is_usable",
    "distance_analysis_is_usable",
    "speed_analysis_is_usable",
    "elevation_analysis_is_usable",
    "elevation_per_distance_analysis_is_usable",
    "power_analysis_is_usable",
    "power_load_analysis_is_usable",
    "stored_tss_is_usable",
    "cp_based_intensity_analysis_is_usable",
    "hr_analysis_is_usable",
    "cadence_analysis_is_usable",
]

print(
    "분석 사용 가능 열 개수:",
    len(analysis_usability_columns)
)

cleaned_df[
    analysis_usability_columns
].sum().sort_values(
    ascending=False
)

분석 사용 가능 열 개수: 12


workout_time_is_usable                       591
riding_time_is_usable                        571
distance_analysis_is_usable                  541
speed_analysis_is_usable                     540
cadence_analysis_is_usable                   527
elevation_analysis_is_usable                 496
stored_tss_is_usable                         490
elevation_per_distance_analysis_is_usable    471
power_analysis_is_usable                     469
power_load_analysis_is_usable                469
hr_analysis_is_usable                        458
cp_based_intensity_analysis_is_usable        379
dtype: int64

In [97]:
final_analysis_columns = (
    base_analysis_columns
    + derived_metric_columns
    + calculation_weight_columns
    + quality_flag_columns
    + analysis_usability_columns
)

print(
    "최종 분석 열 개수:",
    len(final_analysis_columns)
)

duplicate_column_count = (
    len(final_analysis_columns)
    - len(set(final_analysis_columns))
)

print(
    "중복된 열 이름 개수:",
    duplicate_column_count
)

최종 분석 열 개수: 63
중복된 열 이름 개수: 0


In [98]:
final_df = cleaned_df[
    final_analysis_columns
].copy()

print("최종 데이터 크기:", final_df.shape)

최종 데이터 크기: (592, 63)


### 최종 분석 열 정리 결과

기본 분석 열 18개, 파생 지표 6개, 계산 가중치 9개, 품질 관련 열
18개와 분석 사용 가능 열 12개를 결합했다. 중복된 열 이름은 없었으며,
최종 데이터프레임 `final_df`는 592행과 63열로 구성되었다.

`final_df`는 원본 주요 지표를 유지하면서 파생 지표와 데이터 품질 정보를
함께 제공한다. 이후 분석에서는 라이드 전체를 일괄 삭제하지 않고 각
`*_is_usable` 열을 이용해 목적에 맞는 기록만 선택한다.


## 23. 정제 데이터 저장 및 검증

최종 데이터프레임을 `data/processed`에 CSV로 저장한다. 저장된 파일을
날짜 열과 함께 다시 불러와 행·열 크기, 열 순서와 자료형이 유지되는지
확인한다. 가공 데이터는 원본에서 다시 생성할 수 있으므로 Git에는
포함하지 않는다.


In [99]:
processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True,
)

processed_path = (
    processed_dir
    / "goldencheetah_bike_rides_cleaned.csv"
)

final_df.to_csv(
    processed_path,
    index=False,
)

print("정제 데이터 저장 경로:", processed_path)

정제 데이터 저장 경로: /Users/hooni/Documents/ChatGPT/Cycling App/data/processed/goldencheetah_bike_rides_cleaned.csv


In [100]:
reloaded_df = pd.read_csv(
    processed_path,
    parse_dates=["date"],
)

print(
    "다시 불러온 데이터 크기:",
    reloaded_df.shape
)

print(
    "열 순서 일치:",
    reloaded_df.columns.tolist()
    == final_df.columns.tolist()
)

print(
    "날짜 자료형:",
    reloaded_df["date"].dtype
)

다시 불러온 데이터 크기: (592, 63)
열 순서 일치: True
날짜 자료형: datetime64[us, UTC]


In [101]:
reloaded_df.dtypes.value_counts()

float64                30
bool                   29
str                     2
datetime64[us, UTC]     1
int64                   1
Name: count, dtype: int64

### 정제 데이터 저장 및 검증 결과

`final_df`를 `data/processed/goldencheetah_bike_rides_cleaned.csv`로
저장했다. 다시 불러온 데이터는 592행과 63열이었고, 원래 데이터와
열 순서도 일치했다. 날짜는 UTC가 포함된 `datetime64[us, UTC]`로
변환되었다. 마이크로초 정밀도는 주간·월간 훈련 집계에 충분하다.

재로딩한 자료형은 실수형 30개, 불리언 29개, 문자열 2개, 정수형
1개와 날짜형 1개로 구성되어 저장 전 열의 의미가 유지되었다.

이로써 GoldenCheetah 원본 JSON의 구조 탐색, 주요 지표 검증, 품질
라벨과 분석 사용 기준 정의, 최종 데이터 생성 및 저장까지 완료했다.
다음 분석은 별도 노트북에서 정제 CSV를 불러와 훈련 흐름을 시각화한다.
